# Mackey–Glass chaos-aware delay-coordinate system identification

This is a revised copy of `MG_FINAL_CLEAN.ipynb`; the source notebook is not modified.

The central correction is methodological: the original model was selected and evaluated on its training embedding, while the attractor check could accept a periodic orbit. This edition uses disjoint train/validation/test windows and tunes the delay, embedding dimension, and normalized ridge penalty against both short-term forecast error and long-run attractor statistics. The selected configuration is:

- embedding delay: **4.5 time units**
- embedding dimension: **6**
- polynomial degree: **3**
- normalized degree-weighted ridge penalty: **$10^{-4}$**

A model is called chaotic only when it stays bounded over long rollouts, preserves the reference distribution and spectrum, and has a reproducibly positive Benettin maximal Lyapunov exponent. Exact long-term pointwise agreement is neither expected nor meaningful for chaotic trajectories; long-term quality is assessed statistically.


## Environment

Recommended Python version: 3.10 or newer.

```bash
python -m pip install numpy pandas matplotlib scipy scikit-learn pysindy
```

The optional new-history-function experiment also requires:

```bash
python -m pip install ddeint
```

For a repository, place this notebook at the project root and commit a matching
`requirements.txt` or environment file.


In [ ]:
from itertools import combinations_with_replacement, product
from pathlib import Path
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    import pysindy as ps
except ImportError:
    ps = None
from scipy.integrate import solve_ivp
from scipy.linalg import eig
from scipy.optimize import fsolve, root
from scipy.signal import find_peaks, welch
from scipy.spatial import cKDTree
from scipy.stats import wasserstein_distance

# Reproducibility and output policy
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The chaos-aware sweep and final diagnostics are enabled in this revision.
RUN_EMBEDDING_DIAGNOSTICS = False
RUN_MODEL_SWEEP = True
RUN_CHAOS_DIAGNOSTICS = True
RUN_ADVANCED_DYNAMICS = False
RUN_EQUILIBRIUM_ANALYSIS = True
RUN_PRUNING_SWEEPS = False
RUN_LIBRARY_COMPARISON = False
RUN_PRUNE_ADD_SWEEP = False
RUN_NOISE_SWEEP = False
RUN_PERIODIC_ORBIT_SEARCH = True

np.seterr(divide="ignore", invalid="ignore", over="ignore")
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


## 1. Numerical configuration

The scalar Mackey-Glass delay differential equation is

\[
\dot{x}(t) =
\frac{\beta x(t-\tau)}{1+x(t-\tau)^n}-\gamma x(t).
\]

The delay-coordinate state is
\(X(t)=[x(t),x(t-\Delta),\ldots,x(t-(d-1)\Delta)]\).


In [ ]:
# Mackey-Glass parameters (unchanged from the source notebook)
BETA = 0.2
GAMMA = 0.1
NONLINEARITY = 10
DDE_DELAY = 17.0
HISTORY_VALUE = 1.2

# Time discretization and genuinely disjoint data windows
DT = 0.1
T_MAX = 5000.0
TRAIN_START = 117.0
TRAIN_END = 417.0
VALIDATION_START = 700.0
VALIDATION_END = 1300.0
TEST_START = 1500.0
TEST_END = 2500.0
ATTRACTOR_REFERENCE_START = 3500.0
ATTRACTOR_REFERENCE_END = 4500.0

# Chaos-aware selected model (chosen by the sweep below)
EMBEDDING_DELAY = 4.5
EMBEDDING_DIMENSION = 6
BASELINE_POLY_DEGREE = 3
BASELINE_RIDGE_ALPHA = 1e-4
RIDGE_DEGREE_WEIGHT = 1.0

# Shared rollout metrics
HORIZON_CAP = 150.0
CUMULATIVE_ERROR_TOLERANCE = 0.05
STATE_BOUNDS = (0.15, 1.65)

TOTAL_STEPS = int(round(T_MAX / DT))
TAU_STEPS = int(round(DDE_DELAY / DT))
TRAIN_START_STEPS = int(round(TRAIN_START / DT))
TRAIN_END_STEPS = int(round(TRAIN_END / DT))
VALIDATION_START_STEPS = int(round(VALIDATION_START / DT))
VALIDATION_END_STEPS = int(round(VALIDATION_END / DT))
TEST_START_STEPS = int(round(TEST_START / DT))
TEST_END_STEPS = int(round(TEST_END / DT))


## 2. Shared utilities

These functions replace the repeated implementations that appeared throughout the
original notebook. Every experiment below uses the same delay-embedding, RK4 rollout,
predictive-horizon, NRMSE, and model-evaluation definitions.


In [ ]:
def generate_mackey_glass_euler(
    beta=BETA,
    gamma=GAMMA,
    exponent=NONLINEARITY,
    delay=DDE_DELAY,
    dt=DT,
    t_max=T_MAX,
    history_value=HISTORY_VALUE,
):
    """Generate x(t) with explicit Euler and a constant prehistory."""
    delay_steps = int(round(delay / dt))
    total_steps = int(round(t_max / dt))
    values = np.full(total_steps + delay_steps + 1, history_value, dtype=float)

    for i in range(delay_steps, delay_steps + total_steps):
        x_now = values[i]
        x_delay = values[i - delay_steps]
        derivative = beta * x_delay / (1.0 + x_delay**exponent) - gamma * x_now
        values[i + 1] = x_now + dt * derivative

    time = np.arange(total_steps + 1) * dt
    series = values[delay_steps : delay_steps + total_steps + 1]
    return time, series


def build_delay_embedding(series, start_step, end_step, delay_steps, dimension):
    """Build aligned columns [x(t), x(t-delay), ..., x(t-(d-1)delay)]."""
    earliest = start_step - (dimension - 1) * delay_steps
    if earliest < 0:
        raise ValueError("The requested embedding reaches before the available series.")
    if end_step > len(series):
        raise ValueError("The requested embedding extends beyond the available series.")
    columns = [
        series[start_step - j * delay_steps : end_step - j * delay_steps]
        for j in range(dimension)
    ]
    return np.column_stack(columns)


def build_polynomial_exponents(state_dimension, max_degree):
    exponents = [np.zeros(state_dimension, dtype=int)]
    for degree in range(1, max_degree + 1):
        for combo in combinations_with_replacement(range(state_dimension), degree):
            exponent = np.zeros(state_dimension, dtype=int)
            for index in combo:
                exponent[index] += 1
            exponents.append(exponent)
    return np.asarray(exponents)


def polynomial_feature_names(powers):
    names = []
    for power in powers:
        terms = []
        for index, exponent in enumerate(power):
            if exponent == 1:
                terms.append(f"x{index}")
            elif exponent > 1:
                terms.append(f"x{index}^{exponent}")
        names.append(" ".join(terms) if terms else "1")
    return np.asarray(names, dtype=object)


class PolynomialLibraryAdapter:
    """Small PySINDy-compatible polynomial library used by the fitted model."""

    def __init__(self, powers):
        self.powers_ = np.asarray(powers, dtype=int)
        self._names = polynomial_feature_names(self.powers_)

    def transform(self, X):
        X = np.atleast_2d(np.asarray(X, dtype=float))
        return np.prod(X[:, None, :] ** self.powers_[None, :, :], axis=2)

    def get_feature_names(self, input_features=None):
        return list(self._names)

    def get_feature_names_out(self, input_features=None):
        return self._names.copy()


class NormalizedRidgePolynomialModel:
    """Polynomial vector field fit after RMS-normalizing library columns."""

    def __init__(self, coefficients, powers):
        self._coefficients = np.asarray(coefficients, dtype=float)
        self.feature_library = PolynomialLibraryAdapter(powers)

    def predict(self, X):
        return self.feature_library.transform(X) @ self._coefficients.T

    def coefficients(self):
        return self._coefficients.copy()

    def print(self, precision=3):
        names = self.feature_library.get_feature_names_out()
        for state_index, row in enumerate(self._coefficients):
            terms = [
                f"{coefficient:+.{precision}f} {name}"
                for coefficient, name in zip(row, names)
                if abs(coefficient) > 1e-12
            ]
            print(f"(x{state_index})' = " + " ".join(terms))


def fit_sindy_model(
    X,
    dt,
    degree=3,
    threshold=0.0,
    alpha=BASELINE_RIDGE_ALPHA,
    degree_weight=RIDGE_DEGREE_WEIGHT,
):
    """Fit a dense polynomial ODE with scale-invariant degree-weighted ridge.

    PySINDy's raw STLSQ alpha is scale-dependent. Normalizing each feature column
    before ridge makes alpha comparable across delays/dimensions and is the tuning
    change that prevents the selected vector field from drifting to a limit cycle.
    """
    X = np.asarray(X, dtype=float)
    Xdot = np.gradient(X, dt, axis=0, edge_order=2)
    powers = build_polynomial_exponents(X.shape[1], degree)
    library = PolynomialLibraryAdapter(powers)
    theta = library.transform(X)
    scales = np.sqrt(np.mean(theta**2, axis=0))
    scales[scales < 1e-14] = 1.0
    theta_normalized = theta / scales

    feature_degree = powers.sum(axis=1)
    penalty = np.maximum(feature_degree, 1) ** degree_weight
    penalty[0] = 0.0  # do not penalize the intercept
    normal_matrix = theta_normalized.T @ theta_normalized + alpha * np.diag(penalty)
    rhs = theta_normalized.T @ Xdot
    try:
        coefficients_normalized = np.linalg.solve(normal_matrix, rhs).T
    except np.linalg.LinAlgError:
        coefficients_normalized = np.linalg.lstsq(normal_matrix, rhs, rcond=1e-12)[0].T
    coefficients = coefficients_normalized / scales[None, :]
    if threshold > 0:
        coefficients[np.abs(coefficients) < threshold] = 0.0
    return NormalizedRidgePolynomialModel(coefficients, powers)


class FastPolynomialModel:
    """Fast NumPy evaluator for a fitted polynomial model."""

    def __init__(self, fitted_model):
        self.feature_library = fitted_model.feature_library
        self._coefficients = np.asarray(fitted_model.coefficients()).copy()
        self._powers = np.asarray(self.feature_library.powers_)

    def predict(self, X):
        X = np.atleast_2d(X)
        theta = np.prod(X[:, None, :] ** self._powers[None, :, :], axis=2)
        return theta @ self._coefficients.T

    def coefficients(self):
        return self._coefficients.copy()


def model_rhs(model, state):
    return np.asarray(model.predict(np.asarray(state)[None, :]))[0]


def rk4_step(model, state, dt):
    k1 = model_rhs(model, state)
    k2 = model_rhs(model, state + 0.5 * dt * k1)
    k3 = model_rhs(model, state + 0.5 * dt * k2)
    k4 = model_rhs(model, state + dt * k3)
    return state + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)


def safe_rk4_rollout(model, x0, times, bounds=STATE_BOUNDS):
    times = np.asarray(times)
    if len(times) < 2:
        raise ValueError("At least two time points are required.")
    dt = float(times[1] - times[0])
    lower, upper = bounds
    trajectory = np.zeros((len(times), len(x0)), dtype=float)
    trajectory[0] = x0
    for i in range(1, len(times)):
        try:
            next_state = rk4_step(model, trajectory[i - 1], dt)
        except Exception:
            return None
        if not np.all(np.isfinite(next_state)) or np.any(next_state < lower) or np.any(next_state > upper):
            return None
        trajectory[i] = next_state
    return trajectory


def predictive_horizon(X_true, X_pred, dt, tolerance=CUMULATIVE_ERROR_TOLERANCE):
    if X_pred is None:
        return np.nan
    n = min(len(X_true), len(X_pred))
    cumulative_error = np.cumsum((X_true[:n, 0] - X_pred[:n, 0]) ** 2) * dt
    crossings = np.flatnonzero(cumulative_error > tolerance)
    return float(crossings[0] * dt) if len(crossings) else float((n - 1) * dt)


def normalized_rmse(X_true, X_pred):
    if X_pred is None:
        return np.nan
    n = min(len(X_true), len(X_pred))
    true = X_true[:n, 0]
    pred = X_pred[:n, 0]
    return float(np.sqrt(np.mean((true - pred) ** 2)) / (np.std(true) + 1e-12))


def evenly_spaced_test_indices(n_samples, horizon_length, n_initial_conditions=20):
    latest = n_samples - horizon_length
    if latest <= 0:
        raise ValueError("The rollout horizon is longer than the available data.")
    return np.linspace(0, latest - 1, min(n_initial_conditions, latest), dtype=int)


def evaluate_model(model, X_reference, times, test_indices, tolerance=CUMULATIVE_ERROR_TOLERANCE, bounds=STATE_BOUNDS):
    horizons, errors = [], []
    stable_rollouts = 0
    for idx in test_indices:
        truth = X_reference[idx : idx + len(times)]
        prediction = safe_rk4_rollout(model, truth[0], times, bounds=bounds)
        if prediction is None:
            continue
        stable_rollouts += 1
        horizons.append(predictive_horizon(truth, prediction, times[1] - times[0], tolerance))
        errors.append(normalized_rmse(truth, prediction))
    return {
        "stable_rollouts": stable_rollouts,
        "stable_fraction": stable_rollouts / len(test_indices),
        "mean_horizon": float(np.mean(horizons)) if horizons else np.nan,
        "median_horizon": float(np.median(horizons)) if horizons else np.nan,
        "mean_nrmse": float(np.mean(errors)) if errors else np.nan,
    }


def spectral_entropy(signal, dt=DT):
    _, power = welch(signal - np.mean(signal), fs=1.0 / dt, nperseg=min(2048, len(signal)))
    power = power[1:]
    power = power / (power.sum() + 1e-30)
    return float(-np.sum(power * np.log(power + 1e-30)) / np.log(len(power)))


def autocorrelation_vector(signal, max_lag_time=100.0, stride=10):
    centered = np.asarray(signal) - np.mean(signal)
    spectrum = np.fft.rfft(centered, n=2 * len(centered))
    acf = np.fft.irfft(spectrum * np.conj(spectrum))[: len(centered)]
    acf /= acf[0] + 1e-30
    stop = min(len(acf), int(round(max_lag_time / DT)) + 1)
    return acf[:stop:stride]


def compare_attractor_statistics(predicted, reference):
    predicted = np.asarray(predicted)
    reference = np.asarray(reference)
    pred_acf = autocorrelation_vector(predicted)
    ref_acf = autocorrelation_vector(reference)
    n = min(len(pred_acf), len(ref_acf))
    return {
        "std_ratio": float(np.std(predicted) / (np.std(reference) + 1e-12)),
        "mean_error_std_units": float(abs(np.mean(predicted) - np.mean(reference)) / (np.std(reference) + 1e-12)),
        "wasserstein_std_units": float(wasserstein_distance(predicted, reference) / (np.std(reference) + 1e-12)),
        "spectral_entropy": spectral_entropy(predicted),
        "reference_spectral_entropy": spectral_entropy(reference),
        "acf_rmse": float(np.sqrt(np.mean((pred_acf[:n] - ref_acf[:n]) ** 2))),
    }


def save_figure(fig, filename, dpi=200):
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    return path


## 3. Generate the Mackey-Glass trajectory and delay embedding

The target coordinate begins one DDE-delay after `TRAIN_START`. This preserves the
alignment convention used by the original notebook while making the effective time
labels explicit. With the default settings, the four coordinate histories begin near
the requested training boundary.


In [ ]:
time_full, data = generate_mackey_glass_euler()

delay_steps = int(round(EMBEDDING_DELAY / DT))
X_delayed = build_delay_embedding(
    data, TRAIN_START_STEPS, TRAIN_END_STEPS, delay_steps, EMBEDDING_DIMENSION
)
X_validation = build_delay_embedding(
    data, VALIDATION_START_STEPS, VALIDATION_END_STEPS, delay_steps, EMBEDDING_DIMENSION
)
X_test = build_delay_embedding(
    data, TEST_START_STEPS, TEST_END_STEPS, delay_steps, EMBEDDING_DIMENSION
)
t_delayed = np.arange(TRAIN_START_STEPS, TRAIN_END_STEPS) * DT
x_t = X_delayed[:, 0]
attractor_reference = data[
    int(round(ATTRACTOR_REFERENCE_START / DT)) : int(round(ATTRACTOR_REFERENCE_END / DT)) + 1
]

print(f"Scalar trajectory shape: {data.shape}")
print(f"Training embedding:   {X_delayed.shape}, t={TRAIN_START:.1f}..{TRAIN_END:.1f}")
print(f"Validation embedding: {X_validation.shape}, t={VALIDATION_START:.1f}..{VALIDATION_END:.1f}")
print(f"Test embedding:       {X_test.shape}, t={TEST_START:.1f}..{TEST_END:.1f}")


## 4. Optional embedding diagnostics

Average mutual information (AMI) suggests a delay, while false nearest neighbors (FNN)
assesses whether the embedding dimension is large enough. These computations are
exploratory and are disabled by default.


In [ ]:
def mutual_information_histogram(x, y, bins=20):
    joint, _, _ = np.histogram2d(x, y, bins=bins)
    pxy = joint / joint.sum()
    px = pxy.sum(axis=1)
    py = pxy.sum(axis=0)
    expected = px[:, None] * py[None, :]
    valid = (pxy > 0) & (expected > 0)
    return float(np.sum(pxy[valid] * np.log(pxy[valid] / expected[valid])))


def average_mutual_information(series, lag_steps, bins=20):
    scores = []
    for lag in lag_steps:
        scores.append(
            mutual_information_histogram(series[:-lag], series[lag:], bins=bins)
        )
    return np.asarray(scores)


def first_local_minimum(values):
    for i in range(1, len(values) - 1):
        if values[i] < values[i - 1] and values[i] < values[i + 1]:
            return i
    return int(np.argmin(values))


def scalar_delay_embedding(series, delay_steps, dimension):
    n = len(series) - dimension * delay_steps
    if n <= 1:
        raise ValueError("Series is too short for the requested embedding.")
    return np.column_stack(
        [series[j * delay_steps : j * delay_steps + n] for j in range(dimension)]
    )


def false_nearest_neighbor_fractions(
    series,
    delay_steps,
    max_dimension=8,
    ratio_threshold=15.0,
    sample_size=1000,
    seed=RANDOM_SEED,
):
    """Estimate the FNN fraction using nearest neighbors in successive embeddings."""
    local_rng = np.random.default_rng(seed)
    full = scalar_delay_embedding(series, delay_steps, max_dimension + 1)
    fractions = []

    for dimension in range(1, max_dimension + 1):
        current = full[:, :dimension]
        expanded = full[:, : dimension + 1]
        sample = local_rng.choice(
            len(current), size=min(sample_size, len(current)), replace=False
        )
        tree = cKDTree(current)
        distances, neighbors = tree.query(current[sample], k=2)
        nearest_distance = distances[:, 1]
        nearest_index = neighbors[:, 1]
        added_distance = np.abs(
            expanded[sample, dimension] - expanded[nearest_index, dimension]
        )
        ratio = added_distance / np.maximum(nearest_distance, 1e-12)
        fractions.append(float(np.mean(ratio > ratio_threshold)))
    return np.asarray(fractions)


if RUN_EMBEDDING_DIAGNOSTICS:
    lag_steps = np.arange(1, 400)
    ami = average_mutual_information(data, lag_steps)
    best_lag_index = first_local_minimum(ami)
    selected_delay_steps = int(lag_steps[best_lag_index])

    fnn = false_nearest_neighbor_fractions(data, selected_delay_steps)
    dimensions = np.arange(1, len(fnn) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(lag_steps * DT, ami, color="steelblue")
    axes[0].axvline(selected_delay_steps * DT, color="crimson", ls="--")
    axes[0].set(xlabel="Lag (t.u.)", ylabel="Average mutual information", title="AMI")

    axes[1].plot(dimensions, 100 * fnn, "o-", color="darkgreen")
    axes[1].axhline(5, color="crimson", ls="--", label="5% threshold")
    axes[1].set(
        xlabel="Embedding dimension",
        ylabel="False nearest neighbors (%)",
        title="FNN diagnostic",
    )
    axes[1].legend()
    plt.tight_layout()
    save_figure(fig, "mg_embedding_diagnostics.png")
    plt.show()

    print(f"First AMI minimum: {selected_delay_steps * DT:.2f} t.u.")
else:
    print("Embedding diagnostics skipped. Set RUN_EMBEDDING_DIAGNOSTICS = True to run.")


## 5. Chaos-aware selected polynomial model

The model below uses the selected hyperparameters and normalized ridge regression. Unlike the source notebook, all reported predictive metrics use later, unseen windows.


In [ ]:
pysindy_model = fit_sindy_model(
    X_delayed,
    dt=DT,
    degree=BASELINE_POLY_DEGREE,
    threshold=0.0,
    alpha=BASELINE_RIDGE_ALPHA,
)

print("Learned chaos-aware model:")
pysindy_model.print()
print(f"\nCoefficient matrix shape: {pysindy_model.coefficients().shape}")
print(
    f"Selected parameters: delay={EMBEDDING_DELAY}, dimension={EMBEDDING_DIMENSION}, "
    f"degree={BASELINE_POLY_DEGREE}, alpha={BASELINE_RIDGE_ALPHA:g}"
)
model = FastPolynomialModel(pysindy_model)


In [ ]:
baseline_times = np.arange(len(X_delayed)) * DT
X_sim_delayed = safe_rk4_rollout(model, X_delayed[0], baseline_times)

if X_sim_delayed is None:
    warnings.warn("The selected rollout left the configured safety bounds.")
else:
    fig = plt.figure(figsize=(12, 5))
    ax1 = fig.add_subplot(121, projection="3d")
    ax1.plot(X_delayed[:, 0], X_delayed[:, 1], X_delayed[:, 2], "r--", lw=0.6, alpha=0.7, label="Mackey-Glass data")
    ax1.plot(X_sim_delayed[:, 0], X_sim_delayed[:, 1], X_sim_delayed[:, 2], "k-", lw=0.7, alpha=0.8, label="learned rollout")
    ax1.set(xlabel="$x(t)$", ylabel=f"$x(t-{EMBEDDING_DELAY:g})$", zlabel=f"$x(t-{2 * EMBEDDING_DELAY:g})$", title="Training attractor")
    ax1.legend(fontsize=8)

    ax2 = fig.add_subplot(122)
    ax2.plot(t_delayed, X_delayed[:, 0], "r--", lw=0.8, label="Mackey-Glass data")
    ax2.plot(t_delayed, X_sim_delayed[:, 0], "k-", lw=0.9, label="learned rollout")
    ax2.set(xlabel="Time", ylabel="$x(t)$", title="Training-window rollout")
    ax2.legend(fontsize=8)
    plt.tight_layout()
    save_figure(fig, "mg_chaos_selected_training_fit.png")
    plt.show()


In [ ]:
horizon_times = np.arange(0.0, HORIZON_CAP + DT, DT)
validation_indices = evenly_spaced_test_indices(
    len(X_validation), len(horizon_times), n_initial_conditions=12
)
validation_metrics = evaluate_model(model, X_validation, horizon_times, validation_indices)
print(pd.Series(validation_metrics, name="held-out validation"))

representative_idx = int(validation_indices[len(validation_indices) // 2])
X_true_example = X_validation[representative_idx : representative_idx + len(horizon_times)]
X_pred_example = safe_rk4_rollout(model, X_true_example[0], horizon_times)

if X_pred_example is not None:
    cumulative_error = np.cumsum((X_true_example[:, 0] - X_pred_example[:, 0]) ** 2) * DT
    horizon = predictive_horizon(X_true_example, X_pred_example, DT)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(horizon_times, X_true_example[:, 0], "r--", lw=1, label="unseen truth")
    axes[0].plot(horizon_times, X_pred_example[:, 0], "k-", lw=1, label="prediction")
    axes[0].set(xlabel="Forecast time", ylabel="$x(t)$", title="Representative held-out forecast")
    axes[0].legend()
    axes[1].plot(horizon_times, cumulative_error, color="darkgreen", lw=2)
    axes[1].axhline(CUMULATIVE_ERROR_TOLERANCE, color="crimson", ls="--", label="tolerance")
    axes[1].axvline(horizon, color="darkorange", ls=":", label=f"horizon = {horizon:.1f}")
    axes[1].set(xlabel="Forecast time", ylabel="Cumulative squared error", title="Predictive-horizon calculation")
    axes[1].legend()
    plt.tight_layout()
    save_figure(fig, "mg_chaos_heldout_forecast.png")
    plt.show()


## 6. Chaos-aware hyperparameter sweep

The sweep uses the training window only for fitting. It ranks models on a separate validation window using short-term NRMSE plus long-run Wasserstein distance, autocorrelation mismatch, spectral-entropy mismatch, and amplitude mismatch. The broad development sweep covered delays 4–7, dimensions 3–6, degrees 2–3, and eight ridge strengths; the compact reproducible sweep below covers the neighborhood of the winner.


In [ ]:
def run_pysindy_model_sweep(
    series,
    delays=(4.0, 4.5, 5.0, 5.5),
    degrees=(3,),
    dimensions=(4, 5, 6),
    alphas=(1e-6, 1e-4, 1e-3, 1e-2),
    n_initial_conditions=3,
):
    rows = []
    combinations = list(product(delays, degrees, dimensions, alphas))
    short_times = np.arange(0.0, 100.0 + DT, DT)
    # A 1,000-unit rollout is long enough to expose the limit-cycle collapse
    # that shorter health checks can miss.
    long_times = np.arange(0.0, 1000.0 + DT, DT)

    for run_number, (delay, degree, dimension, alpha) in enumerate(combinations, start=1):
        started = time.perf_counter()
        lag = int(round(delay / DT))
        try:
            X_train_candidate = build_delay_embedding(series, TRAIN_START_STEPS, TRAIN_END_STEPS, lag, dimension)
            X_validation_candidate = build_delay_embedding(series, VALIDATION_START_STEPS, VALIDATION_END_STEPS, lag, dimension)
            candidate = fit_sindy_model(X_train_candidate, DT, degree=degree, alpha=alpha)
            candidate_fast = FastPolynomialModel(candidate)
            indices = evenly_spaced_test_indices(len(X_validation_candidate), len(short_times), n_initial_conditions)
            metrics = evaluate_model(candidate_fast, X_validation_candidate, short_times, indices)
            long_trajectory = safe_rk4_rollout(candidate_fast, X_validation_candidate[500], long_times)
            if long_trajectory is None:
                attractor = {"std_ratio": np.nan, "wasserstein_std_units": np.nan, "spectral_entropy": np.nan, "reference_spectral_entropy": np.nan, "acf_rmse": np.nan}
                score = np.inf
            else:
                attractor = compare_attractor_statistics(long_trajectory[2000:, 0], attractor_reference)
                entropy_error = abs(attractor["spectral_entropy"] - attractor["reference_spectral_entropy"])
                if metrics["stable_fraction"] == 1.0:
                    score = (
                        metrics["mean_nrmse"]
                        + 1.5 * attractor["wasserstein_std_units"]
                        + 1.5 * attractor["acf_rmse"]
                        + 3.0 * entropy_error
                        + 0.5 * abs(np.log(max(attractor["std_ratio"], 1e-8)))
                    )
                else:
                    score = np.inf
            rows.append({
                "delay": delay, "degree": degree, "dimension": dimension, "alpha": alpha,
                **metrics, **attractor, "selection_score": score,
                "elapsed_seconds": time.perf_counter() - started,
            })
        except Exception as exc:
            rows.append({"delay": delay, "degree": degree, "dimension": dimension, "alpha": alpha, "selection_score": np.inf, "error": str(exc)})

    return pd.DataFrame(rows).sort_values("selection_score", ascending=True, ignore_index=True)


if RUN_MODEL_SWEEP:
    df_model_sweep = run_pysindy_model_sweep(data)
    df_model_sweep.to_csv(OUTPUT_DIR / "mg_chaos_aware_model_sweep.csv", index=False)
    display(df_model_sweep.head(12))
    print("Configured winner:", {"delay": EMBEDDING_DELAY, "dimension": EMBEDDING_DIMENSION, "degree": BASELINE_POLY_DEGREE, "alpha": BASELINE_RIDGE_ALPHA})
else:
    print("Chaos-aware sweep skipped. Set RUN_MODEL_SWEEP = True to run.")


## 7. Long-run chaos and truly held-out test diagnostics

This section directly checks the failure mode described in the request. The selected model is compared with the original `(delay=6, dimension=4, degree=3, alpha=0)` fit on a later test window. Positive finite-time divergence alone is not accepted: the surrogate must remain bounded, preserve attractor statistics, and produce a positive maximal Lyapunov estimate across multiple initial conditions.


In [ ]:
def estimate_maximal_lyapunov(
    model,
    x0,
    dt,
    total_time=600.0,
    transient_time=200.0,
    renormalization_time=1.0,
    perturbation=1e-7,
    bounds=STATE_BOUNDS,
    seed=RANDOM_SEED,
):
    """Benettin estimate with repeated renormalization and a long burn-in."""
    local_rng = np.random.default_rng(seed)
    lower, upper = bounds
    reference = np.asarray(x0, dtype=float).copy()
    for _ in range(int(round(transient_time / dt))):
        reference = rk4_step(model, reference, dt)
        if not np.all(np.isfinite(reference)) or np.any(reference < lower) or np.any(reference > upper):
            return np.nan, {"status": "failed during transient"}

    direction = local_rng.normal(size=len(reference))
    direction /= np.linalg.norm(direction)
    perturbed = reference + perturbation * direction
    block_steps = max(1, int(round(renormalization_time / dt)))
    block_time = block_steps * dt
    n_blocks = int(round(total_time / block_time))
    cumulative_growth = 0.0
    times, running = [], []

    for block in range(n_blocks):
        for _ in range(block_steps):
            reference = rk4_step(model, reference, dt)
            perturbed = rk4_step(model, perturbed, dt)
            if (
                not np.all(np.isfinite(reference)) or not np.all(np.isfinite(perturbed))
                or np.any(reference < lower) or np.any(reference > upper)
                or np.any(perturbed < lower) or np.any(perturbed > upper)
            ):
                return np.nan, {"status": "failed during rollout", "times": np.asarray(times), "running_mle": np.asarray(running)}
        delta = perturbed - reference
        delta_norm = np.linalg.norm(delta)
        if not np.isfinite(delta_norm) or delta_norm <= 0:
            return np.nan, {"status": "invalid perturbation"}
        cumulative_growth += np.log(delta_norm / perturbation)
        elapsed = (block + 1) * block_time
        times.append(elapsed)
        running.append(cumulative_growth / elapsed)
        perturbed = reference + perturbation * delta / delta_norm

    return float(cumulative_growth / (n_blocks * block_time)), {
        "status": "completed", "times": np.asarray(times), "running_mle": np.asarray(running)
    }


def stress_test_model(label, fitted_model, X_reference, long_reference):
    fast_model = FastPolynomialModel(fitted_model)
    test_times = np.arange(0.0, HORIZON_CAP + DT, DT)
    test_indices = evenly_spaced_test_indices(len(X_reference), len(test_times), 12)
    forecast = evaluate_model(fast_model, X_reference, test_times, test_indices)

    long_times = np.arange(0.0, 2000.0 + DT, DT)
    attractor_rows, mle_values = [], []
    for j, index in enumerate([0, 1000, 3000]):
        long_rollout = safe_rk4_rollout(fast_model, X_reference[index], long_times)
        if long_rollout is not None:
            attractor_rows.append(compare_attractor_statistics(long_rollout[5000:, 0], long_reference))
        for seed in [11 + j, 31 + j]:
            mle, _ = estimate_maximal_lyapunov(fast_model, X_reference[index], DT, seed=seed)
            if np.isfinite(mle):
                mle_values.append(mle)

    return {
        "model": label,
        **forecast,
        "stable_long_rollouts": len(attractor_rows),
        "mean_wasserstein": float(np.mean([r["wasserstein_std_units"] for r in attractor_rows])) if attractor_rows else np.nan,
        "mean_std_ratio": float(np.mean([r["std_ratio"] for r in attractor_rows])) if attractor_rows else np.nan,
        "mean_spectral_entropy": float(np.mean([r["spectral_entropy"] for r in attractor_rows])) if attractor_rows else np.nan,
        "reference_spectral_entropy": spectral_entropy(long_reference),
        "mle_mean": float(np.mean(mle_values)) if mle_values else np.nan,
        "mle_std": float(np.std(mle_values)) if mle_values else np.nan,
        "mle_successes": len(mle_values),
    }


if RUN_CHAOS_DIAGNOSTICS:
    original_X_train = build_delay_embedding(data, TRAIN_START_STEPS, TRAIN_END_STEPS, int(round(6.0 / DT)), 4)
    original_X_test = build_delay_embedding(data, TEST_START_STEPS, TEST_END_STEPS, int(round(6.0 / DT)), 4)
    original_fit = fit_sindy_model(original_X_train, DT, degree=3, alpha=0.0)

    comparison = pd.DataFrame([
        stress_test_model("original baseline", original_fit, original_X_test, attractor_reference),
        stress_test_model("chaos-aware selected", pysindy_model, X_test, attractor_reference),
    ]).set_index("model")
    display(comparison.T)
    comparison.to_csv(OUTPUT_DIR / "mg_chaos_stress_test.csv")

    mle, mle_diagnostics = estimate_maximal_lyapunov(model, X_test[1000], DT, seed=31)
    long_times = np.arange(0.0, 2000.0 + DT, DT)
    long_selected = safe_rk4_rollout(model, X_test[1000], long_times)
    if long_selected is None:
        raise RuntimeError("Selected model failed its long-run stability check.")
    long_stats = compare_attractor_statistics(long_selected[5000:, 0], attractor_reference)
    print(f"Selected-model MLE: {mle:.6f} ({mle_diagnostics['status']})")
    print("Selected long-run statistics:", long_stats)
    if not (np.isfinite(mle) and mle > 0.001):
        warnings.warn("The final MLE did not pass the positive-chaos threshold.")

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes[0, 0].plot(long_times, long_selected[:, 0], color="black", lw=0.45)
    axes[0, 0].set(xlabel="Time", ylabel="$x(t)$", title="2,000-unit learned rollout (bounded, non-collapsing)")
    axes[0, 1].plot(long_selected[:, 0], long_selected[:, 1], color="black", lw=0.35, alpha=0.7)
    axes[0, 1].set(xlabel="$x(t)$", ylabel=f"$x(t-{EMBEDDING_DELAY:g})$", title="Learned long-run attractor")
    axes[1, 0].hist(attractor_reference, bins=60, density=True, alpha=0.55, color="crimson", label="true")
    axes[1, 0].hist(long_selected[5000:, 0], bins=60, density=True, alpha=0.55, color="black", label="learned")
    axes[1, 0].set(xlabel="$x$", ylabel="Density", title="Invariant-distribution comparison")
    axes[1, 0].legend()
    axes[1, 1].plot(mle_diagnostics["times"], mle_diagnostics["running_mle"], color="darkgreen")
    axes[1, 1].axhline(mle, color="crimson", ls="--", label=f"final MLE = {mle:.4f}")
    axes[1, 1].axhline(0.0, color="black", lw=0.8)
    axes[1, 1].set(xlabel="Accumulation time", ylabel="Running MLE", title="Benettin MLE convergence")
    axes[1, 1].legend()
    plt.tight_layout()
    save_figure(fig, "mg_chaos_long_run_diagnostics.png")
    plt.show()
else:
    print("Chaos diagnostics skipped. Set RUN_CHAOS_DIAGNOSTICS = True to run.")


### 7.1 Very-long-time limit-cycle exclusion test

The preceding 2,000-unit test is extended here to **20,000 time units** from three test-set initial conditions. Only the final 5,000 units are used for recurrence diagnostics. A stable limit cycle would produce near-perfect recurrence at one lag, a small set of repeated peak heights, and a largest Lyapunov exponent near zero after a long burn-in. The selected model is required to show the opposite behavior while retaining stationary variance and spectral complexity.


In [ ]:
def limit_cycle_diagnostics(signal, reference):
    signal = np.asarray(signal)
    centered = signal - np.mean(signal)
    spectrum = np.fft.rfft(centered, n=2 * len(centered))
    acf = np.fft.irfft(spectrum * np.conj(spectrum))[: len(centered)]
    acf /= acf[0] + 1e-30
    lag_start = int(round(10.0 / DT))
    lag_stop = min(len(acf), int(round(200.0 / DT)) + 1)
    best_lag = lag_start + int(np.argmax(acf[lag_start:lag_stop]))
    recurrence_nrmse = float(
        np.sqrt(np.mean((signal[best_lag:] - signal[:-best_lag]) ** 2))
        / (np.std(signal) + 1e-30)
    )
    peaks, _ = find_peaks(signal, distance=int(round(2.0 / DT)), prominence=0.005)
    peak_values = signal[peaks]
    return {
        "std_ratio_to_true": float(np.std(signal) / np.std(reference)),
        "spectral_entropy": spectral_entropy(signal),
        "best_recurrence_lag": float(best_lag * DT),
        "best_recurrence_acf": float(acf[best_lag]),
        "best_recurrence_nrmse": recurrence_nrmse,
        "late_peak_count": int(len(peak_values)),
        "unique_peak_heights_1e-4": int(len(np.unique(np.round(peak_values, 4)))),
    }


very_long_times = np.arange(0.0, 20000.0 + DT, DT)
very_long_rows = []
for run_id, test_index in enumerate([0, 1000, 3000], start=1):
    trajectory = safe_rk4_rollout(model, X_test[test_index], very_long_times)
    if trajectory is None:
        very_long_rows.append({"run": run_id, "bounded": False})
        continue

    final_window = trajectory[-int(round(5000.0 / DT)) :, 0]
    diagnostics = limit_cycle_diagnostics(final_window, attractor_reference)
    window_stds, window_entropies = [], []
    window_steps = int(round(5000.0 / DT))
    for window_id in range(4):
        window = trajectory[window_id * window_steps : (window_id + 1) * window_steps, 0]
        window_stds.append(float(np.std(window)))
        window_entropies.append(spectral_entropy(window))

    post_burn_mle, _ = estimate_maximal_lyapunov(
        model,
        X_test[test_index],
        DT,
        transient_time=5000.0,
        total_time=2000.0,
        renormalization_time=1.0,
        seed=100 + run_id,
    )
    very_long_rows.append({
        "run": run_id,
        "bounded": True,
        "duration": 20000.0,
        "post_5000_mle": post_burn_mle,
        "window_std_minmax_ratio": min(window_stds) / max(window_stds),
        "window_entropy_min": min(window_entropies),
        **diagnostics,
    })

very_long_results = pd.DataFrame(very_long_rows).set_index("run")
display(very_long_results.T)
very_long_results.to_csv(OUTPUT_DIR / "mg_very_long_limit_cycle_test.csv")

limit_cycle_excluded = bool(
    very_long_results["bounded"].all()
    and (very_long_results["post_5000_mle"] > 0.001).all()
    and (very_long_results["window_std_minmax_ratio"] > 0.95).all()
    and (very_long_results["best_recurrence_nrmse"] > 0.20).all()
    and (very_long_results["unique_peak_heights_1e-4"] > 50).all()
)
print("Very-long-time limit-cycle exclusion:", "PASS" if limit_cycle_excluded else "FAIL")
if not limit_cycle_excluded:
    warnings.warn("One or more very-long-time chaos criteria failed.")


## 8. Equilibrium benchmark and advanced dynamics

The equilibrium benchmark runs by default on the selected cubic forecast model,
independently of `RUN_ADVANCED_DYNAMICS`. It compares learned roots against all
three constant DDE equilibria and records delay-coordinate consistency and local
spectra. The finite ODE spectrum is not the full infinite DDE spectrum. The search
uses 256 seeded random starts, 20 training states, and the three exact constant
embeddings; it is not exhaustive. The historical `n_keep=1` rule selects the root
closest to a sampled training state; the full root list remains visible. The older history/manifold analysis remains opt-in.


In [ ]:
def fit_polynomial_history(
    learned_trajectory,
    delay=DDE_DELAY,
    dt=DT,
    minimum_degree=3,
    maximum_degree=29,
    error_tolerance=1e-5,
):
    """Fit a polynomial to one delay-length window of a learned trajectory."""
    n_window = int(round(delay / dt))
    window_time = np.arange(n_window) * dt
    window_values = np.asarray(learned_trajectory)[:n_window, 0]

    best_polynomial, best_degree, best_error = None, None, np.inf
    for degree in range(minimum_degree, maximum_degree + 1):
        coefficients = np.polyfit(window_time, window_values, degree)
        polynomial = np.poly1d(coefficients)
        error = float(np.max(np.abs(polynomial(window_time) - window_values)))
        best_polynomial, best_degree, best_error = polynomial, degree, error
        if error < error_tolerance:
            break
    return best_polynomial, best_degree, best_error


def make_history_function(polynomial, delay=DDE_DELAY):
    """Map t in [-delay, 0] to the fitted polynomial's [0, delay] interval."""
    def history(t):
        shifted = np.clip(t + delay, 0.0, delay)
        return float(polynomial(shifted))
    return history


def simulate_mg_with_history(
    history,
    times,
    beta=BETA,
    gamma=GAMMA,
    exponent=NONLINEARITY,
    delay=DDE_DELAY,
):
    """Integrate the true Mackey-Glass DDE with ddeint and a supplied history."""
    try:
        from ddeint import ddeint
    except ImportError as exc:
        raise ImportError(
            "This optional experiment requires `python -m pip install ddeint`."
        ) from exc

    def right_hand_side(Y, t):
        x_now = Y(t)
        x_delay = Y(t - delay)
        return beta * x_delay / (1.0 + x_delay**exponent) - gamma * x_now

    solution = ddeint(right_hand_side, history, np.asarray(times))
    return solution[:, 0] if solution.ndim > 1 else solution


if RUN_ADVANCED_DYNAMICS:
    if X_sim_delayed is None:
        raise RuntimeError("A bounded baseline rollout is required for the history experiment.")

    history_polynomial, history_degree, history_error = fit_polynomial_history(
        X_sim_delayed
    )
    history_function = make_history_function(history_polynomial)
    fresh_times = np.arange(0.0, 200.0, DT)
    true_from_fitted_history = simulate_mg_with_history(
        history_function, fresh_times
    )

    ideal_embedded_ic = np.array(
        [
            history_function(-index * EMBEDDING_DELAY)
            for index in range(EMBEDDING_DIMENSION)
        ]
    )
    nearest_index = int(
        np.argmin(np.linalg.norm(X_delayed - ideal_embedded_ic, axis=1))
    )
    safe_embedded_ic = X_delayed[nearest_index]
    learned_from_history = safe_rk4_rollout(
        model, safe_embedded_ic, fresh_times
    )

    print(
        f"History polynomial degree={history_degree}, "
        f"max fitting error={history_error:.3e}, "
        f"nearest embedded-data index={nearest_index}"
    )

    if learned_from_history is not None:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(
            fresh_times,
            true_from_fitted_history,
            "r--",
            lw=0.9,
            label="True MG with fitted history",
        )
        ax.plot(
            fresh_times,
            learned_from_history[:, 0],
            "k-",
            lw=0.9,
            label="PySINDy rollout",
        )
        ax.set(
            xlabel="Time",
            ylabel="$x(t)$",
            title="New-history-function trajectory comparison",
        )
        ax.legend()
        save_figure(fig, "mg_new_history_comparison.png")
        plt.show()


In [ ]:
def find_equilibria(model, dimension, n_starts=2000, bounds=(0.0, 2.0), seed=RANDOM_SEED):
    """Find and deduplicate roots of the learned vector field."""
    local_rng = np.random.default_rng(seed)

    def field(z):
        return model_rhs(model, z)

    roots = []
    for guess in local_rng.uniform(bounds[0], bounds[1], size=(n_starts, dimension)):
        try:
            solution, info, status, _ = fsolve(field, guess, full_output=True)
            if status != 1 or np.linalg.norm(field(solution)) >= 1e-8:
                continue
            if all(np.linalg.norm(solution - existing) > 1e-4 for existing in roots):
                roots.append(solution)
        except Exception:
            continue
    return roots


def numerical_jacobian(model, point, epsilon=1e-6):
    """Centered finite-difference Jacobian of a learned vector field."""
    point = np.asarray(point)
    jacobian = np.zeros((len(point), len(point)))
    for j in range(len(point)):
        perturbation = np.zeros(len(point))
        perturbation[j] = epsilon
        jacobian[:, j] = (
            model_rhs(model, point + perturbation)
            - model_rhs(model, point - perturbation)
        ) / (2 * epsilon)
    return jacobian


def classify_equilibrium(eigenvalues, tolerance=1e-10):
    real_parts = np.real(eigenvalues)
    if np.all(real_parts < -tolerance):
        return "sink"
    if np.all(real_parts > tolerance):
        return "source"
    if np.any(real_parts < -tolerance) and np.any(real_parts > tolerance):
        return "saddle"
    return "non-hyperbolic"


def analyze_equilibria(model, roots, X_reference):
    records = []
    for point in roots:
        jacobian = numerical_jacobian(model, point)
        eigenvalues, eigenvectors = eig(jacobian)
        records.append(
            {
                "z_star": point,
                "jacobian": jacobian,
                "eigenvalues": eigenvalues,
                "eigenvectors": eigenvectors,
                "classification": classify_equilibrium(eigenvalues),
                "min_distance_to_attractor": float(
                    np.min(np.linalg.norm(X_reference - point, axis=1))
                ),
            }
        )
    return sorted(records, key=lambda record: record["min_distance_to_attractor"])


def dde_characteristic_roots(
    gamma,
    linearized_delayed_coefficient,
    delay,
    real_range=(-1.0, 0.2),
    imag_range=(-2.0, 2.0),
    grid=(20, 40),
):
    """Numerically solve lambda + gamma - a exp(-lambda tau) = 0."""
    def residual(z):
        lam = z[0] + 1j * z[1]
        value = lam + gamma - linearized_delayed_coefficient * np.exp(-lam * delay)
        return [value.real, value.imag]

    roots = []
    for real_guess in np.linspace(*real_range, grid[0]):
        for imag_guess in np.linspace(*imag_range, grid[1]):
            solution = root(residual, [real_guess, imag_guess])
            if not solution.success:
                continue
            lam = solution.x[0] + 1j * solution.x[1]
            if all(abs(lam - existing) >= 1e-5 for existing in roots):
                roots.append(lam)
    return sorted(roots, key=lambda value: value.real, reverse=True)


if RUN_ADVANCED_DYNAMICS:
    equilibrium_roots = find_equilibria(model, X_delayed.shape[1])
    equilibria = analyze_equilibria(model, equilibrium_roots, X_delayed)

    print(f"Found {len(equilibria)} distinct learned equilibria.")
    for i, record in enumerate(equilibria):
        print(
            f"{i}: z*={np.round(record['z_star'], 5)}, "
            f"class={record['classification']}, "
            f"distance={record['min_distance_to_attractor']:.4f}"
        )
        for value in sorted(record["eigenvalues"], key=lambda item: item.real):
            print(f"    {value.real:+.6f} {value.imag:+.6f}i")

    leading_dde_roots = dde_characteristic_roots(
        gamma=GAMMA,
        linearized_delayed_coefficient=-0.4,
        delay=DDE_DELAY,
    )
    print("\nLeading DDE characteristic roots:")
    for value in leading_dde_roots[:20]:
        print(value)

    if equilibria:
        selected_equilibrium = equilibria[0]
        fig = plt.figure(figsize=(13, 5.5))
        ax = fig.add_subplot(121, projection="3d")
        ax.plot(
            X_delayed[:, 0],
            X_delayed[:, 1],
            X_delayed[:, 2],
            "r--",
            lw=0.5,
            alpha=0.4,
            label="Attractor",
        )
        point = selected_equilibrium["z_star"]
        ax.scatter(*point[:3], s=140, color="black", label="Nearest equilibrium")
        ax.set(
            xlabel="$x(t)$",
            ylabel=f"$x(t-{EMBEDDING_DELAY:g})$",
            zlabel=f"$x(t-{2 * EMBEDDING_DELAY:g})$",
            title="Nearest learned equilibrium",
        )
        ax.legend()

        spectrum_ax = fig.add_subplot(122)
        dde_spectrum = np.asarray(leading_dde_roots[:12])
        ode_spectrum = np.asarray(selected_equilibrium["eigenvalues"])
        spectrum_ax.scatter(
            dde_spectrum.real,
            dde_spectrum.imag,
            s=55,
            label="DDE characteristic roots",
        )
        spectrum_ax.scatter(
            ode_spectrum.real,
            ode_spectrum.imag,
            marker="x",
            s=70,
            label="Learned ODE eigenvalues",
        )
        spectrum_ax.axvline(0.0, color="black", ls="--", lw=0.8)
        spectrum_ax.axhline(0.0, color="black", ls="--", lw=0.8)
        spectrum_ax.set(
            xlabel="Real part",
            ylabel="Imaginary part",
            title="Local spectrum comparison",
        )
        spectrum_ax.legend(fontsize=8)

        plt.tight_layout()
        save_figure(fig, "mg_equilibrium_and_spectrum.png")
        plt.show()
else:
    print("Advanced dynamics skipped. Set RUN_ADVANCED_DYNAMICS = True to run.")


### Experimental two-dimensional local manifolds

The following refactors the separate stable and unstable manifold blocks into one
implementation. It preserves the original truncated-series construction. Treat the
result as an exploratory local approximation; it is not used by the pruning or
forecasting results.


In [ ]:
def compute_local_manifold(
    model,
    equilibrium,
    direction="unstable",
    order=20,
    scale=0.01,
    parameter_range=0.3,
    grid_size=40,
):
    """Construct the original notebook's truncated 2-D manifold approximation."""
    point = equilibrium["z_star"]
    eigenvalues, eigenvectors = eig(equilibrium["jacobian"])
    ordering = np.argsort(eigenvalues.real)
    if direction == "unstable":
        ordering = ordering[::-1]
    elif direction != "stable":
        raise ValueError("direction must be 'stable' or 'unstable'")

    lam1, lam2 = eigenvalues[ordering[:2]]
    vec1, vec2 = eigenvectors[:, ordering[0]], eigenvectors[:, ordering[1]]
    dimension = len(point)

    coefficients = np.zeros((order + 1, order + 1, dimension), dtype=complex)
    coefficients[0, 0] = point
    coefficients[1, 0] = scale * vec1
    coefficients[0, 1] = scale * vec2
    jacobian = numerical_jacobian(model, point)

    def convolution_rhs(m_index, n_index):
        rhs = np.zeros(dimension, dtype=complex)
        for state_index in range(dimension):
            total = 0.0j
            for p in range(m_index):
                r = m_index - 1 - p
                for q in range(n_index):
                    s = n_index - 1 - q
                    if (p > 0 or q > 0) and (r > 0 or s > 0):
                        total += (
                            coefficients[p, q, 0]
                            * coefficients[r, s, state_index]
                        )
            rhs[state_index] = total
        return rhs

    for total_order in range(2, order + 1):
        for m_index in range(1, total_order):
            n_index = total_order - m_index
            if m_index < n_index:
                coefficients[m_index, n_index] = np.conj(
                    coefficients[n_index, m_index]
                )
                continue
            lhs = jacobian - (m_index * lam1 + n_index * lam2) * np.eye(dimension)
            try:
                coefficients[m_index, n_index] = np.linalg.solve(
                    lhs, convolution_rhs(m_index, n_index)
                )
            except np.linalg.LinAlgError:
                coefficients[m_index, n_index] = 0.0

    parameters = np.linspace(-parameter_range, parameter_range, grid_size)
    manifold = np.zeros((grid_size, grid_size, dimension))
    for i, u in enumerate(parameters):
        for j, v in enumerate(parameters):
            sigma1, sigma2 = u + 1j * v, u - 1j * v
            value = np.zeros(dimension, dtype=complex)
            for m_index in range(order + 1):
                for n_index in range(order + 1 - m_index):
                    value += (
                        coefficients[m_index, n_index]
                        * sigma1**m_index
                        * sigma2**n_index
                    )
            manifold[i, j] = value.real

    return {
        "surface": manifold,
        "coefficients": coefficients,
        "eigenvalues": (lam1, lam2),
        "direction": direction,
    }


def evaluate_manifold_series(coefficients, u, v):
    """Evaluate a truncated manifold series at one parameter-space point."""
    sigma1, sigma2 = u + 1j * v, u - 1j * v
    order = coefficients.shape[0] - 1
    value = np.zeros(coefficients.shape[-1], dtype=complex)
    for m_index in range(order + 1):
        for n_index in range(order + 1 - m_index):
            value += (
                coefficients[m_index, n_index]
                * sigma1**m_index
                * sigma2**n_index
            )
    return value.real


def validate_local_manifold(model, manifold_result, duration=20.0):
    """Integrate sample points forward (unstable) or backward (stable)."""
    parameter_points = [
        (0.05, 0.05),
        (-0.05, 0.05),
        (0.05, -0.05),
        (-0.05, -0.05),
        (0.05, 0.0),
        (0.0, 0.05),
        (-0.05, 0.0),
        (0.0, -0.05),
    ]
    direction = manifold_result["direction"]
    end_time = duration if direction == "unstable" else -duration

    def ode_rhs(_, state):
        return model_rhs(model, state)

    trajectories = []
    for u, v in parameter_points:
        initial_state = evaluate_manifold_series(
            manifold_result["coefficients"], u, v
        )
        try:
            solution = solve_ivp(
                ode_rhs,
                (0.0, end_time),
                initial_state,
                max_step=0.05,
                rtol=1e-8,
                atol=1e-10,
            )
            if solution.success:
                trajectories.append(solution.y.T)
        except Exception:
            continue
    return trajectories


if RUN_ADVANCED_DYNAMICS and equilibria:
    local_unstable_manifold = compute_local_manifold(
        model,
        equilibria[0],
        direction="unstable",
        order=40,
        scale=0.1,
        parameter_range=0.7,
    )
    surface = local_unstable_manifold["surface"]
    validation_trajectories = validate_local_manifold(
        model, local_unstable_manifold
    )

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(
        surface[:, :, 0],
        surface[:, :, 1],
        surface[:, :, 2],
        alpha=0.5,
        cmap="plasma",
        edgecolor="none",
    )
    ax.plot(
        X_delayed[:, 0],
        X_delayed[:, 1],
        X_delayed[:, 2],
        "r--",
        lw=0.4,
        alpha=0.3,
    )
    for trajectory in validation_trajectories:
        ax.plot(
            trajectory[:, 0],
            trajectory[:, 1],
            trajectory[:, 2],
            color="darkgreen",
            lw=0.9,
            alpha=0.7,
        )
    ax.set_title("Experimental local unstable manifold")
    save_figure(fig, "mg_experimental_unstable_manifold.png")
    plt.show()


## 9. Shared pruning utilities

`CoefficientModel` evaluates a PySINDy feature library with a supplied coefficient
matrix. It supports both entry-level masked refitting and feature-level zeroing without
duplicating the rollout and metric code.


In [ ]:
class CoefficientModel:
    """A fitted feature library paired with an explicit coefficient matrix."""

    def __init__(self, feature_library, coefficients):
        self.feature_library = feature_library
        self._coefficients = np.asarray(coefficients).copy()

    def predict(self, X):
        X = np.atleast_2d(X)
        theta = np.asarray(self.feature_library.transform(X))
        return theta @ self._coefficients.T

    def coefficients(self):
        return self._coefficients.copy()


def refit_with_mask(base_model, X, Xdot, mask, ridge=1e-10):
    """Refit every state equation using only coefficient entries allowed by mask."""
    theta = np.asarray(base_model.feature_library.transform(X))
    Xdot = np.asarray(Xdot)
    coefficients = np.zeros_like(mask, dtype=float)

    for equation in range(mask.shape[0]):
        keep = mask[equation]
        if not np.any(keep):
            continue
        design = theta[:, keep]
        lhs = design.T @ design + ridge * np.eye(design.shape[1])
        rhs = design.T @ Xdot[:, equation]
        coefficients[equation, keep] = np.linalg.solve(lhs, rhs)
    return coefficients


def masked_refit_model(base_model, X, Xdot, n_prune, ranking="contribution"):
    coefficients = np.asarray(base_model.coefficients())
    if ranking == "coefficient":
        score = np.abs(coefficients)
    elif ranking == "contribution":
        theta = np.asarray(base_model.feature_library.transform(X))
        feature_rms = np.sqrt(np.mean(theta**2, axis=0))
        score = np.abs(coefficients) * feature_rms[None, :]
    else:
        raise ValueError("ranking must be 'coefficient' or 'contribution'")

    mask = np.ones_like(coefficients, dtype=bool)
    prune_order = np.argsort(score, axis=None)
    mask.ravel()[prune_order[:n_prune]] = False
    refit = refit_with_mask(base_model, X, Xdot, mask)
    return CoefficientModel(base_model.feature_library, refit), mask


def feature_pruned_model(base_model, n_prune):
    """Zero whole library features ranked by coefficient L2 norm; do not refit."""
    coefficients = np.asarray(base_model.coefficients()).copy()
    order = np.argsort(np.linalg.norm(coefficients, axis=0))
    coefficients[:, order[:n_prune]] = 0.0
    return CoefficientModel(base_model.feature_library, coefficients)


def pruning_row(model_candidate, X, times, indices):
    metrics = evaluate_model(model_candidate, X, times, indices)
    coefficients = np.asarray(model_candidate.coefficients())
    return {
        **metrics,
        "n_active_entries": int(np.count_nonzero(coefficients)),
        "n_active_features": int(
            np.count_nonzero(np.linalg.norm(coefficients, axis=0))
        ),
    }


### 9.1 Masked-refit coefficient pruning

Coefficient entries are ranked by their estimated contribution
\(|\xi_{ij}|\,\mathrm{RMS}(\Theta_j)\), removed, and the remaining entries are refit.


In [ ]:
def run_masked_refit_sweep(base_model, X, dt):
    Xdot = np.gradient(X, dt, axis=0, edge_order=2)
    n_entries = np.asarray(base_model.coefficients()).size
    prune_counts = [
        count
        for count in (0, 1, 2, 3, 4, 5, 8, 10, 12, 15, 20, 25, 30, 35,
                      40, 50, 60, 70, 80, 90, 100, 110, 120)
        if count < n_entries
    ]
    rows, models = [], {}
    for count in prune_counts:
        candidate, _ = masked_refit_model(
            base_model, X, Xdot, count, ranking="contribution"
        )
        row = {"n_pruned_entries": count, **pruning_row(
            candidate, X, horizon_times, baseline_test_indices
        )}
        rows.append(row)
        models[count] = candidate
        print(row)
    return pd.DataFrame(rows), models


if RUN_PRUNING_SWEEPS:
    df_masked_refit, masked_refit_models = run_masked_refit_sweep(model, X_delayed, DT)
    df_masked_refit.to_csv(
        OUTPUT_DIR / "mg_masked_refit_pruning_sweep.csv", index=False
    )

    plot_horizon = df_masked_refit["mean_horizon"].clip(upper=HORIZON_CAP)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(df_masked_refit["n_pruned_entries"], plot_horizon, "o-", color="darkgreen")
    axes[0].set(xlabel="Pruned coefficient entries", ylabel="Mean horizon", title="Horizon")
    axes[1].plot(
        df_masked_refit["n_pruned_entries"],
        df_masked_refit["mean_nrmse"],
        "s-",
        color="crimson",
    )
    axes[1].set(xlabel="Pruned coefficient entries", ylabel="Mean NRMSE", title="NRMSE")
    axes[2].plot(
        df_masked_refit["n_active_entries"], plot_horizon, "D-", color="steelblue"
    )
    axes[2].invert_xaxis()
    axes[2].set(
        xlabel="Active coefficient entries",
        ylabel="Mean horizon",
        title="Parsimony-horizon tradeoff",
    )
    plt.tight_layout()
    save_figure(fig, "mg_masked_refit_pruning_sweep.png")
    plt.show()
else:
    print("Pruning sweeps skipped. Set RUN_PRUNING_SWEEPS = True to run.")


### 9.2 Feature-level magnitude pruning

Entire library columns are ranked by coefficient-vector norm and zeroed without
refitting. This differs from entry-level masking because a feature is removed from all
state equations at once.


In [ ]:
def run_feature_pruning_sweep(base_model, X):
    n_features = np.asarray(base_model.coefficients()).shape[1]
    prune_counts = [
        count
        for count in (0, 1, 2, 3, 4, 5, 8, 10, 12, 15, 20, 25, 30, 35)
        if count < n_features
    ]
    rows, models = [], {}
    for count in prune_counts:
        candidate = feature_pruned_model(base_model, count)
        row = {"n_pruned_features": count, **pruning_row(
            candidate, X, horizon_times, baseline_test_indices
        )}
        rows.append(row)
        models[count] = candidate
        print(row)
    return pd.DataFrame(rows), models


if RUN_PRUNING_SWEEPS:
    df_feature_pruning, feature_pruned_models = run_feature_pruning_sweep(
        model, X_delayed
    )
    df_feature_pruning.to_csv(
        OUTPUT_DIR / "mg_magnitude_pruning_sweep.csv", index=False
    )

    plot_horizon = df_feature_pruning["mean_horizon"].clip(upper=HORIZON_CAP)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(
        df_feature_pruning["n_pruned_features"],
        plot_horizon,
        "o-",
        color="darkgreen",
    )
    axes[0].set(xlabel="Pruned features", ylabel="Mean horizon", title="Horizon")
    axes[1].plot(
        df_feature_pruning["n_pruned_features"],
        df_feature_pruning["mean_nrmse"],
        "s-",
        color="crimson",
    )
    axes[1].set(xlabel="Pruned features", ylabel="Mean NRMSE", title="NRMSE")
    axes[2].plot(
        df_feature_pruning["n_active_features"],
        plot_horizon,
        "D-",
        color="steelblue",
    )
    axes[2].invert_xaxis()
    axes[2].set(
        xlabel="Active features",
        ylabel="Mean horizon",
        title="Parsimony-horizon tradeoff",
    )
    plt.tight_layout()
    save_figure(fig, "mg_magnitude_pruning_sweep.png")
    plt.show()


## 10. Polynomial and Fourier library comparison

Candidate libraries are fit and evaluated on the same data, rollout starts, bounds,
horizon definition, and NRMSE definition.


In [ ]:
def make_fourier_library(periods, harmonics=(1,), include_bias=True):
    functions, names = [], []

    for period in periods:
        for harmonic in harmonics:
            omega = 2 * np.pi * harmonic / period
            functions.extend(
                [
                    lambda x, omega=omega: np.sin(omega * x),
                    lambda x, omega=omega: np.cos(omega * x),
                ]
            )
            names.extend(
                [
                    lambda x, h=harmonic, p=period: f"sin(2pi*{h}/{p:.4g}*{x})",
                    lambda x, h=harmonic, p=period: f"cos(2pi*{h}/{p:.4g}*{x})",
                ]
            )

    return ps.CustomLibrary(
        library_functions=functions,
        function_names=names,
        include_bias=include_bias,
        interaction_only=True,
    )


def make_candidate_library(kind, poly_degree=2, periods=(2 * np.pi,), harmonics=(1,)):
    if kind == "polynomial":
        return ps.PolynomialLibrary(degree=poly_degree, include_bias=True)
    if kind == "fourier":
        return make_fourier_library(periods, harmonics, include_bias=True)
    if kind == "polynomial_fourier":
        return ps.ConcatLibrary(
            [
                ps.PolynomialLibrary(degree=poly_degree, include_bias=True),
                make_fourier_library(periods, harmonics, include_bias=False),
            ]
        )
    raise ValueError(f"Unknown library kind: {kind}")


def run_library_comparison(X, dt):
    Xdot = np.gradient(X, dt, axis=0, edge_order=2)
    amplitude_range = np.ptp(X[:, 0])
    period_sets = {
        "range": (amplitude_range,),
        "half_range": (0.5 * amplitude_range,),
        "double_range": (2.0 * amplitude_range,),
    }
    harmonic_sets = {"k1": (1,), "k1_k2": (1, 2), "k1_k2_k3": (1, 2, 3)}
    rows, models = [], {}

    for kind in ("polynomial", "fourier", "polynomial_fourier"):
        periods_to_try = {"none": (2 * np.pi,)} if kind == "polynomial" else period_sets
        harmonics_to_try = {"none": (1,)} if kind == "polynomial" else harmonic_sets

        for period_label, periods in periods_to_try.items():
            for harmonic_label, harmonics in harmonics_to_try.items():
                tag = (kind, period_label, harmonic_label)
                try:
                    library = make_candidate_library(
                        kind, poly_degree=2, periods=periods, harmonics=harmonics
                    )
                    candidate = ps.SINDy(
                        feature_library=library,
                        optimizer=ps.STLSQ(threshold=0.0, alpha=0.0),
                    )
                    candidate.fit(X, t=dt, x_dot=Xdot)
                    metrics = evaluate_model(
                        candidate, X, horizon_times, baseline_test_indices
                    )
                    coefficients = np.asarray(candidate.coefficients())
                    row = {
                        "library": kind,
                        "period_label": period_label,
                        "harmonics": str(harmonics),
                        **metrics,
                        "n_features": coefficients.shape[1],
                        "n_active_entries": int(np.count_nonzero(coefficients)),
                        "cond_theta": float(np.linalg.cond(library.transform(X))),
                    }
                    rows.append(row)
                    models[tag] = candidate
                    print(row)
                except Exception as exc:
                    print(f"Skipped {tag}: {exc}")

    return pd.DataFrame(rows).sort_values(
        ["stable_fraction", "mean_horizon", "mean_nrmse"],
        ascending=[False, False, True],
    ), models


if RUN_LIBRARY_COMPARISON:
    df_library_comparison, library_models = run_library_comparison(X_delayed, DT)
    df_library_comparison.to_csv(
        OUTPUT_DIR / "mg_library_comparison.csv", index=False
    )
    display(df_library_comparison)
else:
    print("Library comparison skipped. Set RUN_LIBRARY_COMPARISON = True to run.")


## 11. Stepwise quadratic pruning and cubic augmentation

Phase A removes the active quadratic monomial with the smallest coefficient norm and
refits the remaining terms. Phase B starts from the pruned model and greedily adds the
cubic monomial that most reduces the training residual. The figure cell implements the
paper caption for `MG_Prune_Add_Sweep.png`.


In [ ]:
def build_polynomial_exponents(state_dimension, max_degree):
    exponents, degrees = [], []
    for degree in range(max_degree + 1):
        for combination in combinations_with_replacement(range(state_dimension), degree):
            exponent = np.zeros(state_dimension, dtype=int)
            for index in combination:
                exponent[index] += 1
            exponents.append(exponent)
            degrees.append(degree)
    return np.asarray(exponents), np.asarray(degrees)


def polynomial_transform(X, exponents):
    X = np.atleast_2d(X)
    return np.prod(X[:, None, :] ** exponents[None, :, :], axis=2)


class PolynomialCoefficientModel:
    def __init__(self, exponents, coefficients):
        self.exponents = np.asarray(exponents)
        self._coefficients = np.asarray(coefficients)

    def predict(self, X):
        return polynomial_transform(X, self.exponents) @ self._coefficients

    def coefficients(self):
        return self._coefficients.T


def fit_active_polynomial(theta, Xdot, active_mask):
    active = np.flatnonzero(active_mask)
    fitted, *_ = np.linalg.lstsq(theta[:, active], Xdot, rcond=None)
    coefficients = np.zeros((theta.shape[1], Xdot.shape[1]))
    coefficients[active] = fitted
    return coefficients


def polynomial_feature_names(exponents):
    names = []
    for exponent in exponents:
        factors = []
        for index, power in enumerate(exponent):
            if power == 1:
                factors.append(f"x{index}")
            elif power > 1:
                factors.append(f"x{index}^{power}")
        names.append("1" if not factors else " ".join(factors))
    return names


def run_prune_add_sweep(X, dt, prune_steps=8, add_steps=8):
    Xdot = np.gradient(X, dt, axis=0, edge_order=2)
    exponents, degrees = build_polynomial_exponents(X.shape[1], max_degree=3)
    names = polynomial_feature_names(exponents)
    theta = polynomial_transform(X, exponents)
    quadratic_or_less = degrees <= 2
    quadratic = degrees == 2
    cubic = degrees == 3

    rows, models = [], {}

    def record(stage, step, mask, coefficients, **action):
        candidate = PolynomialCoefficientModel(exponents, coefficients)
        metrics = evaluate_model(
            candidate, X, horizon_times, baseline_test_indices
        )
        residual = float(np.linalg.norm(theta @ coefficients - Xdot))
        row = {
            "step": step,
            "stage": stage,
            "n_active_features": int(mask.sum()),
            "cond_theta": float(np.linalg.cond(theta[:, mask])),
            "train_residual": residual,
            **metrics,
            **action,
        }
        rows.append(row)
        models[step] = candidate
        print(row)

    active_mask = quadratic_or_less.copy()
    coefficients = fit_active_polynomial(theta, Xdot, active_mask)
    record("baseline_quadratic", 0, active_mask, coefficients)

    for step in range(1, prune_steps + 1):
        norms = np.linalg.norm(coefficients, axis=1)
        candidates = np.where(active_mask & quadratic, norms, np.inf)
        weakest = int(np.argmin(candidates))
        if not np.isfinite(candidates[weakest]):
            break
        active_mask[weakest] = False
        coefficients = fit_active_polynomial(theta, Xdot, active_mask)
        record(
            "prune_quadratic",
            step,
            active_mask,
            coefficients,
            removed_feature=names[weakest],
        )

    remaining_cubic = list(np.flatnonzero(cubic & ~active_mask))
    for addition in range(1, add_steps + 1):
        best_feature, best_residual, best_coefficients = None, np.inf, None
        for feature in remaining_cubic:
            trial_mask = active_mask.copy()
            trial_mask[feature] = True
            trial_coefficients = fit_active_polynomial(theta, Xdot, trial_mask)
            residual = np.linalg.norm(theta @ trial_coefficients - Xdot)
            if residual < best_residual:
                best_feature = feature
                best_residual = residual
                best_coefficients = trial_coefficients

        if best_feature is None:
            break
        active_mask[best_feature] = True
        remaining_cubic.remove(best_feature)
        coefficients = best_coefficients
        record(
            "add_cubic",
            prune_steps + addition,
            active_mask,
            coefficients,
            added_feature=names[best_feature],
        )

    return pd.DataFrame(rows), models


def plot_prune_add_sweep(results):
    """Create the three-panel figure described by the paper caption."""
    baseline = results.iloc[0]
    prune = results[results["stage"] == "prune_quadratic"]
    add = results[results["stage"] == "add_cubic"]
    metrics = [
        ("mean_horizon", "Mean predictive horizon (t.u.)", False),
        ("mean_nrmse", "Mean NRMSE", False),
        ("cond_theta", r"Library condition number $\kappa(\Theta)$", True),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
    marker_scale = 260.0

    for ax, (metric, ylabel, log_scale) in zip(axes, metrics):
        ax.axhline(
            baseline[metric],
            color="0.55",
            ls=":",
            lw=1.5,
            label="Dense quadratic baseline",
        )
        ax.scatter(
            baseline["step"],
            baseline[metric],
            s=marker_scale * baseline["stable_fraction"],
            color="0.35",
            edgecolor="black",
            lw=0.6,
            zorder=4,
        )

        ax.plot(
            np.r_[baseline["step"], prune["step"]],
            np.r_[baseline[metric], prune[metric]],
            color="forestgreen",
            lw=2,
        )
        ax.scatter(
            prune["step"],
            prune[metric],
            s=marker_scale * prune["stable_fraction"],
            color="forestgreen",
            edgecolor="black",
            lw=0.5,
            label="Quadratic pruning",
            zorder=4,
        )

        if len(add):
            phase_b_start = prune.iloc[-1] if len(prune) else baseline
            ax.plot(
                np.r_[phase_b_start["step"], add["step"]],
                np.r_[phase_b_start[metric], add[metric]],
                color="rebeccapurple",
                lw=2,
            )
            ax.scatter(
                add["step"],
                add[metric],
                s=marker_scale * add["stable_fraction"],
                color="rebeccapurple",
                edgecolor="black",
                lw=0.5,
                label="Greedy cubic addition",
                zorder=4,
            )

        ax.set(xlabel="Sweep step", ylabel=ylabel)
        ax.set_xticks(results["step"])
        if log_scale:
            ax.set_yscale("log")

    axes[0].set_title("Predictive horizon")
    axes[1].set_title("Rollout error")
    axes[2].set_title("Regression conditioning")
    axes[0].legend(fontsize=8)
    return fig


if RUN_PRUNE_ADD_SWEEP:
    df_prune_add, prune_add_models = run_prune_add_sweep(X_delayed, DT)
    df_prune_add.to_csv(
        OUTPUT_DIR / "mg_prune_quadratic_add_cubic_sweep.csv", index=False
    )
    figure = plot_prune_add_sweep(df_prune_add)
    save_figure(figure, "MG_Prune_Add_Sweep.png", dpi=300)
    plt.show()
else:
    print("Prune/add sweep skipped. Set RUN_PRUNE_ADD_SWEEP = True to run.")


## 12. Measurement-noise robustness

Gaussian noise is added only to the training series. Each noisy model is evaluated
against a clean held-out trajectory, isolating the effect of identification noise from
evaluation noise.


In [ ]:
def run_noise_robustness_sweep(
    clean_series,
    noise_levels=(0.0, 0.005, 0.01, 0.02, 0.05, 0.10, 0.20),
    repeats=5,
    degree=3,
):
    test_start = TRAIN_END_STEPS + TAU_STEPS
    test_end = TEST_END_STEPS
    X_test_clean = build_delay_embedding(
        clean_series,
        test_start,
        test_end,
        delay_steps,
        EMBEDDING_DIMENSION,
    )
    test_times = np.arange(0.0, 40.0 + DT, DT)
    test_indices = evenly_spaced_test_indices(
        len(X_test_clean), len(test_times), n_initial_conditions=12
    )

    rows = []
    for noise_fraction in noise_levels:
        for repeat in range(repeats):
            local_rng = np.random.default_rng(
                int(1_000_000 * noise_fraction) + repeat
            )
            noisy_series = clean_series + local_rng.normal(
                0.0,
                noise_fraction * np.std(clean_series),
                size=clean_series.shape,
            )
            X_train_noisy = build_delay_embedding(
                noisy_series,
                target_start_step,
                target_end_step,
                delay_steps,
                EMBEDDING_DIMENSION,
            )

            try:
                noisy_model = fit_sindy_model(
                    X_train_noisy, DT, degree=degree
                )
                metrics = evaluate_model(
                    FastPolynomialModel(noisy_model),
                    X_test_clean,
                    test_times,
                    test_indices,
                    bounds=(0.0, 3.0),
                )
            except Exception as exc:
                metrics = {
                    "stable_rollouts": 0,
                    "stable_fraction": 0.0,
                    "mean_horizon": np.nan,
                    "mean_nrmse": np.nan,
                }
                print(f"Noise={noise_fraction}, repeat={repeat} failed: {exc}")

            rows.append(
                {
                    "noise_fraction": noise_fraction,
                    "repeat": repeat,
                    **metrics,
                }
            )
    return pd.DataFrame(rows)


if RUN_NOISE_SWEEP:
    df_noise = run_noise_robustness_sweep(data)
    df_noise.to_csv(OUTPUT_DIR / "mg_noise_results.csv", index=False)
    noise_summary = (
        df_noise.groupby("noise_fraction")
        .agg(
            mean_horizon=("mean_horizon", "mean"),
            std_horizon=("mean_horizon", "std"),
            mean_nrmse=("mean_nrmse", "mean"),
            std_nrmse=("mean_nrmse", "std"),
            mean_stable_fraction=("stable_fraction", "mean"),
        )
        .reset_index()
    )
    display(noise_summary)

    x_percent = 100 * noise_summary["noise_fraction"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].errorbar(
        x_percent,
        noise_summary["mean_horizon"],
        yerr=noise_summary["std_horizon"],
        marker="o",
        color="steelblue",
    )
    axes[0].set(
        xlabel="Noise level (% of signal std)",
        ylabel="Mean predictive horizon (t.u.)",
        title="Horizon vs. measurement noise",
    )
    axes[1].errorbar(
        x_percent,
        noise_summary["mean_nrmse"],
        yerr=noise_summary["std_nrmse"],
        marker="o",
        color="steelblue",
    )
    axes[1].set(
        xlabel="Noise level (% of signal std)",
        ylabel="Mean NRMSE",
        title="NRMSE vs. measurement noise",
    )
    plt.tight_layout()
    save_figure(fig, "mg_noise_robustness.png")
    plt.show()
else:
    print("Noise sweep skipped. Set RUN_NOISE_SWEEP = True to run.")


## 13. Periodic-orbit proposal and true-DDE validation

This opt-in experiment leaves the single-window chaos-aware model used above unchanged. For orbit proposal only, it fits the same six-dimensional cubic polynomial model on five mutually disjoint 300-time-unit windows, using a normalized ridge penalty of $5\times10^{-5}$. These proposal windows do not overlap the original test window $[1500,2500]$ or attractor-reference window $[3500,4500]$. The separate proposal fit is therefore not substituted into the forecasting claims made earlier in the notebook.

The search targets the shortest available nontrivial recurrence in the predeclared 15--80 time-unit period band. Shorter periods reduce amplification in the subsequent true-DDE shooting diagnostic and are the appropriate first targets for a computer-assisted-proof pipeline. The learned orbit is transferred to the true Mackey–Glass DDE by global Fourier collocation, continued through 40, 80, 120, 160, and 200 harmonics, and validated through a separate adaptive DOP853 method-of-steps integration.

The reported `true_defect_before_correction` remains the full one-period shooting defect. The table also records the local true-DDE equation residual and full-waveform correction, which distinguish model mismatch from instability amplification. Acceptance requires a nonconstant orbit, true-equation RMS residual below $10^{-10}$, independently recomputed full-history closure below $10^{-9}$, and a waveform correction below 25%.


### 13.1 Fourier-collocation and independent true-DDE validation utilities


In [ ]:
"""Periodic-orbit proposal and true-DDE validation utilities for Mackey--Glass.

The learned delay-coordinate model is a finite-dimensional autonomous ODE, so its
candidate orbits are refined by ordinary single shooting.  The state of the true
Mackey--Glass DDE is a history function on [-tau, 0].  For the transfer step that
history is represented by a short Fourier series and corrected by shooting the
entire history segment through one candidate period.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
from scipy.spatial import cKDTree


def scan_periodic_recurrences(
    trajectory,
    dt,
    min_period=20.0,
    max_period=250.0,
    search_stride=5,
    neighbors=80,
    max_candidates=30,
):
    """Find and rank close returns of a learned trajectory with a KD tree."""
    trajectory = np.asarray(trajectory, dtype=float)
    sampled_indices = np.arange(0, len(trajectory), search_stride)
    sampled = trajectory[sampled_indices]
    scale = np.std(sampled, axis=0)
    scale[scale < 1e-10] = 1.0
    standardized = (sampled - np.mean(sampled, axis=0)) / scale
    tree = cKDTree(standardized)
    query_count = min(neighbors + 1, len(standardized))
    distances, neighbor_indices = tree.query(
        standardized, k=query_count, workers=-1
    )

    minimum_lag = int(round(min_period / dt))
    maximum_lag = int(round(max_period / dt))
    raw = []
    for local_start, (row_distances, row_neighbors) in enumerate(
        zip(distances, neighbor_indices)
    ):
        start = int(sampled_indices[local_start])
        for distance, local_end in zip(row_distances[1:], row_neighbors[1:]):
            end = int(sampled_indices[int(local_end)])
            if end <= start:
                continue
            lag = end - start
            if minimum_lag <= lag <= maximum_lag:
                raw.append((float(distance), start, end))

    raw.sort(key=lambda item: item[0])
    rows = []
    start_tolerance = max(1, int(round(2.0 / dt)))
    for standardized_distance, start, end in raw:
        period = (end - start) * dt
        duplicate = any(
            abs(start - old["start_index"]) <= start_tolerance
            and abs(period - old["period"]) <= 2.0
            for old in rows
        )
        if duplicate:
            continue

        segment = trajectory[start : end + 1]
        return_error = float(np.linalg.norm(trajectory[end] - trajectory[start]))
        diameter = float(np.max(np.linalg.norm(segment - segment[0], axis=1)))
        rows.append(
            {
                "start_index": start,
                "end_index": end,
                "period": float(period),
                "standardized_distance": standardized_distance,
                "return_error": return_error,
                "segment_diameter": diameter,
                "relative_return_error": float(
                    return_error / (diameter + 1e-12)
                ),
            }
        )
        if len(rows) >= max_candidates:
            break

    columns = [
        "start_index",
        "end_index",
        "period",
        "standardized_distance",
        "return_error",
        "segment_diameter",
        "relative_return_error",
    ]
    if not rows:
        return pd.DataFrame(columns=columns)
    return pd.DataFrame(rows).sort_values(
        ["relative_return_error", "standardized_distance"],
        ignore_index=True,
    )


def adaptive_endpoint(
    initial_state,
    period,
    rhs_function,
    rtol=1e-9,
    atol=1e-11,
    max_step=0.1,
    max_abs=10.0,
):
    """High-accuracy endpoint of the learned autonomous ODE."""
    initial_state = np.asarray(initial_state, dtype=float)
    solution = solve_ivp(
        lambda _time, state: rhs_function(state),
        (0.0, float(period)),
        initial_state,
        method="DOP853",
        t_eval=[float(period)],
        rtol=rtol,
        atol=atol,
        max_step=max_step,
    )
    if (
        not solution.success
        or solution.y.shape[1] == 0
        or not np.all(np.isfinite(solution.y[:, -1]))
        or np.max(np.abs(solution.y[:, -1])) > max_abs
    ):
        return np.full_like(initial_state, np.nan)
    return solution.y[:, -1]


def periodic_closure_defect(state, endpoint):
    """Normalized full-state periodicity defect."""
    state = np.asarray(state, dtype=float)
    endpoint = np.asarray(endpoint, dtype=float)
    scale = max(np.linalg.norm(state - np.mean(state)), 1e-12)
    return float(np.linalg.norm(endpoint - state) / scale)


@dataclass
class PeriodicOrbitRefinement:
    state: np.ndarray
    period: float
    endpoint: np.ndarray
    optimizer_success: bool
    converged: bool
    nfev: int
    cost: float
    relative_closure_defect: float
    time_phase_defect: float
    relative_state_correction: float
    speed_norm: float
    message: str


def refine_periodic_orbit(
    reference_state,
    period_guess,
    endpoint_function,
    rhs_function,
    max_nfev=80,
    defect_tolerance=1e-8,
    period_window=(0.7, 1.3),
    state_bounds=(0.0, 2.0),
):
    """Single-shoot a learned periodic orbit with one time-phase condition."""
    reference_state = np.asarray(reference_state, dtype=float)
    n_state = len(reference_state)
    time_tangent = np.asarray(rhs_function(reference_state), dtype=float)
    state_scale = max(
        np.linalg.norm(reference_state - np.mean(reference_state)), 1e-8
    )
    time_scale = max(state_scale * np.linalg.norm(time_tangent), 1e-10)
    initial = np.concatenate([reference_state, [np.log(period_guess)]])
    lower = np.full_like(initial, -np.inf)
    upper = np.full_like(initial, np.inf)
    lower[:n_state] = state_bounds[0]
    upper[:n_state] = state_bounds[1]
    lower[-1] = np.log(period_window[0] * period_guess)
    upper[-1] = np.log(period_window[1] * period_guess)

    def unpack(parameters):
        return parameters[:-1], float(np.exp(parameters[-1]))

    def residual(parameters):
        state, period = unpack(parameters)
        endpoint = endpoint_function(state, period)
        if not np.all(np.isfinite(endpoint)):
            return np.full(n_state + 1, 1e6)
        closure = (endpoint - state) / state_scale
        displacement = state - reference_state
        time_phase = float(np.dot(displacement, time_tangent) / time_scale)
        return np.concatenate([closure, [time_phase]])

    result = least_squares(
        residual,
        initial,
        bounds=(lower, upper),
        max_nfev=max_nfev,
        xtol=1e-10,
        ftol=1e-10,
        gtol=1e-10,
        x_scale="jac",
        verbose=0,
    )
    state, period = unpack(result.x)
    endpoint = endpoint_function(state, period)
    relative_defect = periodic_closure_defect(state, endpoint)
    displacement = state - reference_state
    time_phase = float(np.dot(displacement, time_tangent) / time_scale)
    state_correction = float(np.linalg.norm(displacement) / state_scale)
    speed_norm = float(np.linalg.norm(rhs_function(state)))
    converged = bool(
        result.success
        and relative_defect < defect_tolerance
        and speed_norm > 1e-6
        and state_correction < 1.0
    )
    return PeriodicOrbitRefinement(
        state=state,
        period=period,
        endpoint=endpoint,
        optimizer_success=bool(result.success),
        converged=converged,
        nfev=int(result.nfev),
        cost=float(result.cost),
        relative_closure_defect=relative_defect,
        time_phase_defect=time_phase,
        relative_state_correction=state_correction,
        speed_norm=speed_norm,
        message=str(result.message),
    )


def sample_learned_periodic_orbit(
    state,
    period,
    rhs_function,
    n_samples=1024,
    rtol=1e-11,
    atol=1e-13,
    max_step=0.05,
):
    """Sample component x(t) over one learned-model period."""
    times = np.linspace(0.0, float(period), n_samples, endpoint=False)
    solution = solve_ivp(
        lambda _time, current: rhs_function(current),
        (0.0, float(period)),
        np.asarray(state, dtype=float),
        method="DOP853",
        t_eval=times,
        rtol=rtol,
        atol=atol,
        max_step=max_step,
    )
    if not solution.success or solution.y.shape[1] != len(times):
        raise RuntimeError("Could not sample the learned periodic orbit.")
    return times, solution.y[0]


def fourier_phase_design(phases, harmonics):
    """Vectorized real Fourier design on phases measured in cycles."""
    phases = np.atleast_1d(np.asarray(phases, dtype=float))
    design = np.empty((len(phases), 2 * harmonics + 1), dtype=float)
    design[:, 0] = 1.0
    harmonic = np.arange(1, harmonics + 1, dtype=float)
    angles = 2.0 * np.pi * phases[:, None] * harmonic[None, :]
    design[:, 1::2] = np.cos(angles)
    design[:, 2::2] = np.sin(angles)
    return design


def fourier_phase_derivative_design(phases, harmonics):
    """Derivative of the Fourier design with respect to cycle phase."""
    phases = np.atleast_1d(np.asarray(phases, dtype=float))
    design = np.zeros((len(phases), 2 * harmonics + 1), dtype=float)
    harmonic = np.arange(1, harmonics + 1, dtype=float)
    angles = 2.0 * np.pi * phases[:, None] * harmonic[None, :]
    angular_frequency = 2.0 * np.pi * harmonic[None, :]
    design[:, 1::2] = -angular_frequency * np.sin(angles)
    design[:, 2::2] = angular_frequency * np.cos(angles)
    return design


def fourier_design(times, period, harmonics):
    """Real Fourier design matrix [1, cos(k wt), sin(k wt)]."""
    return fourier_phase_design(
        np.asarray(times, dtype=float) / float(period), harmonics
    )


def fit_periodic_history(times, values, period, harmonics=12):
    """Least-squares Fourier representation of a periodic scalar waveform."""
    design = fourier_design(times, period, harmonics)
    return np.linalg.lstsq(design, np.asarray(values, dtype=float), rcond=None)[0]


def evaluate_periodic_history(times, coefficients, period):
    coefficients = np.asarray(coefficients, dtype=float)
    harmonics = (len(coefficients) - 1) // 2
    return fourier_design(times, period, harmonics) @ coefficients


def evaluate_periodic_history_derivative(times, coefficients, period):
    coefficients = np.asarray(coefficients, dtype=float)
    times = np.asarray(times, dtype=float)
    harmonics = (len(coefficients) - 1) // 2
    phase_derivative = fourier_phase_derivative_design(
        times / float(period), harmonics
    )
    return phase_derivative @ coefficients / float(period)


def _solve_mackey_glass_by_steps(
    history_coefficients,
    period,
    beta,
    gamma,
    exponent,
    delay,
    max_step=0.05,
    rtol=1e-12,
    atol=1e-14,
):
    """Integrate the true DDE with adaptive DOP853 method-of-steps segments."""
    period = float(period)
    history_coefficients = np.asarray(history_coefficients, dtype=float)

    def history_value(query_time):
        return float(
            evaluate_periodic_history(
                [query_time], history_coefficients, period
            )[0]
        )

    pieces = []

    def completed_value(query_time):
        if query_time <= 1e-14:
            return history_value(query_time)
        for start, end, solution in pieces:
            if query_time <= end + 1e-11:
                clipped = min(max(query_time, start), end)
                return float(solution.sol(clipped)[0])
        raise RuntimeError("Delayed query lies outside completed DDE segments.")

    initial_value = float(
        evaluate_periodic_history([0.0], history_coefficients, period)[0]
    )
    start = 0.0
    while start < period - 1e-13:
        end = min(start + delay, period)

        def derivative(time_value, state_value):
            delayed = completed_value(time_value - delay)
            return [
                beta * delayed / (1.0 + delayed**exponent)
                - gamma * state_value[0]
            ]

        solution = solve_ivp(
            derivative,
            (start, end),
            [initial_value],
            method="DOP853",
            dense_output=True,
            rtol=rtol,
            atol=atol,
            max_step=max_step,
        )
        if not solution.success or not np.all(np.isfinite(solution.y)):
            raise RuntimeError(f"True-DDE integration failed: {solution.message}")
        pieces.append((start, end, solution))
        initial_value = float(solution.y[0, -1])
        start = end
    return pieces, history_value


def _evaluate_mackey_glass_solution(query_times, pieces, history_value):
    """Evaluate a completed method-of-steps solution and its prehistory."""
    query_times = np.asarray(query_times, dtype=float)
    values = np.empty_like(query_times)
    for index, query_time in np.ndenumerate(query_times):
        if query_time <= 0.0:
            values[index] = history_value(float(query_time))
            continue
        for start, end, solution in pieces:
            if query_time <= end + 1e-11:
                clipped = min(max(float(query_time), start), end)
                values[index] = float(solution.sol(clipped)[0])
                break
        else:
            raise RuntimeError("Query lies beyond the integrated DDE interval.")
    return values


def integrate_mackey_glass_history(
    history_coefficients,
    period,
    beta,
    gamma,
    exponent,
    delay,
    max_step=0.05,
    rtol=1e-12,
    atol=1e-14,
):
    """Return a uniformly sampled adaptive method-of-steps DDE solution."""
    pieces, history_value = _solve_mackey_glass_by_steps(
        history_coefficients,
        period,
        beta,
        gamma,
        exponent,
        delay,
        max_step=max_step,
        rtol=rtol,
        atol=atol,
    )
    n_steps = max(1, int(np.ceil(float(period) / max_step)))
    times = np.linspace(0.0, float(period), n_steps + 1)
    values = _evaluate_mackey_glass_solution(times, pieces, history_value)
    return times, values


def mackey_glass_history_closure(
    history_coefficients,
    period,
    beta,
    gamma,
    exponent,
    delay,
    history_nodes=49,
    max_step=0.05,
    rtol=1e-12,
    atol=1e-14,
):
    """Independently shoot one period and compare the full DDE history state."""
    relative_times = np.linspace(-delay, 0.0, history_nodes)
    initial_history = evaluate_periodic_history(
        relative_times, history_coefficients, period
    )
    pieces, history_value = _solve_mackey_glass_by_steps(
        history_coefficients,
        period,
        beta,
        gamma,
        exponent,
        delay,
        max_step=max_step,
        rtol=rtol,
        atol=atol,
    )
    endpoint_times = period + relative_times
    endpoint_history = _evaluate_mackey_glass_solution(
        endpoint_times, pieces, history_value
    )
    scale = max(
        np.linalg.norm(initial_history - np.mean(initial_history)), 1e-12
    )
    defect = float(np.linalg.norm(endpoint_history - initial_history) / scale)
    return initial_history, endpoint_history, defect


def mackey_glass_equation_residual(
    history_coefficients,
    period,
    beta,
    gamma,
    exponent,
    delay,
    n_nodes=4096,
    phase_offset=0.173,
):
    """Evaluate the true DDE equation residual on an independent phase grid."""
    phases = (np.arange(n_nodes, dtype=float) + phase_offset) / n_nodes
    times = phases * float(period)
    values = evaluate_periodic_history(times, history_coefficients, period)
    delayed = evaluate_periodic_history(
        times - delay, history_coefficients, period
    )
    derivatives = evaluate_periodic_history_derivative(
        times, history_coefficients, period
    )
    return (
        derivatives
        - beta * delayed / (1.0 + delayed**exponent)
        + gamma * values
    )


@dataclass
class DDEOrbitRefinement:
    history_coefficients: np.ndarray
    period: float
    initial_history: np.ndarray
    endpoint_history: np.ndarray
    optimizer_success: bool
    converged: bool
    nfev: int
    cost: float
    relative_closure_defect: float
    time_phase_defect: float
    relative_history_correction: float
    relative_waveform_correction: float
    equation_rms_residual: float
    equation_max_residual: float
    amplitude: float
    message: str


def refine_mackey_glass_periodic_orbit(
    seed_coefficients,
    period_guess,
    beta,
    gamma,
    exponent,
    delay,
    history_nodes=129,
    max_step=0.05,
    max_nfev=100,
    defect_tolerance=1e-9,
    period_window=(0.7, 1.3),
    collocation_harmonics=200,
    collocation_oversampling=4,
):
    """Correct a periodic waveform by global Fourier collocation.

    Long single shooting is ill-conditioned for this unstable, roughly
    200-time-unit orbit.  Collocation enforces the true DDE at all phases at
    once; an adaptive method-of-steps integration then validates closure by an
    independent numerical route.
    """
    seed_coefficients = np.asarray(seed_coefficients, dtype=float)
    coefficient_count = 2 * collocation_harmonics + 1
    if len(seed_coefficients) < coefficient_count:
        seed_coefficients = np.pad(
            seed_coefficients, (0, coefficient_count - len(seed_coefficients))
        )
    elif len(seed_coefficients) > coefficient_count:
        seed_coefficients = seed_coefficients[:coefficient_count]

    relative_times = np.linspace(-delay, 0.0, history_nodes)
    seed_history = evaluate_periodic_history(
        relative_times, seed_coefficients, period_guess
    )
    seed_tangent = evaluate_periodic_history_derivative(
        relative_times, seed_coefficients, period_guess
    )
    history_scale = max(
        np.linalg.norm(seed_history - np.mean(seed_history)), 1e-8
    )
    collocation_nodes = max(
        1024, collocation_oversampling * coefficient_count
    )
    phases = (np.arange(collocation_nodes, dtype=float) + 0.37) / collocation_nodes
    collocation_times = phases * period_guess
    value_design = fourier_phase_design(phases, collocation_harmonics)
    phase_derivative_design = fourier_phase_derivative_design(
        phases, collocation_harmonics
    )
    seed_waveform = value_design @ seed_coefficients
    seed_waveform_tangent = (
        phase_derivative_design @ seed_coefficients / period_guess
    )
    equation_scale = max(
        float(np.sqrt(np.mean(seed_waveform_tangent**2))), 1e-3
    )
    time_scale = max(
        np.linalg.norm(seed_waveform_tangent) * np.std(seed_waveform), 1e-10
    )
    initial = np.concatenate([seed_coefficients, [np.log(period_guess)]])
    lower = np.full_like(initial, -np.inf)
    upper = np.full_like(initial, np.inf)
    lower[-1] = np.log(period_window[0] * period_guess)
    upper[-1] = np.log(period_window[1] * period_guess)

    def unpack(parameters):
        return parameters[:-1], float(np.exp(parameters[-1]))

    def residual(parameters):
        coefficients, period = unpack(parameters)
        values = value_design @ coefficients
        delayed_design = fourier_phase_design(
            phases - delay / period, collocation_harmonics
        )
        delayed = delayed_design @ coefficients
        derivatives = phase_derivative_design @ coefficients / period
        equation = (
            derivatives
            - beta * delayed / (1.0 + delayed**exponent)
            + gamma * values
        )
        displacement = values - seed_waveform
        time_phase = float(
            np.dot(displacement, seed_waveform_tangent) / time_scale
        )
        return np.concatenate([equation / equation_scale, [time_phase]])

    result = least_squares(
        residual,
        initial,
        bounds=(lower, upper),
        max_nfev=max_nfev,
        jac="2-point",
        xtol=2e-14,
        ftol=2e-14,
        gtol=2e-14,
        x_scale="jac",
        verbose=0,
    )
    coefficients, period = unpack(result.x)
    initial_history, endpoint_history, relative_defect = (
        mackey_glass_history_closure(
            coefficients,
            period,
            beta,
            gamma,
            exponent,
            delay,
            history_nodes=history_nodes,
            max_step=max_step,
        )
    )
    corrected_waveform = evaluate_periodic_history(
        phases * period, coefficients, period
    )
    corrected_seed_history = evaluate_periodic_history(
        relative_times, seed_coefficients, period_guess
    )
    displacement = initial_history - corrected_seed_history
    waveform_displacement = corrected_waveform - seed_waveform
    time_phase = float(
        np.dot(waveform_displacement, seed_waveform_tangent) / time_scale
    )
    history_correction = float(np.linalg.norm(displacement) / history_scale)
    waveform_scale = max(
        np.linalg.norm(seed_waveform - np.mean(seed_waveform)), 1e-8
    )
    waveform_correction = float(
        np.linalg.norm(waveform_displacement) / waveform_scale
    )
    equation_residual = mackey_glass_equation_residual(
        coefficients,
        period,
        beta,
        gamma,
        exponent,
        delay,
        n_nodes=max(4096, 8 * coefficient_count),
    )
    equation_rms = float(np.sqrt(np.mean(equation_residual**2)))
    equation_max = float(np.max(np.abs(equation_residual)))
    amplitude = float(np.ptp(initial_history))
    converged = bool(
        result.success
        and relative_defect < defect_tolerance
        and equation_rms < 1e-10
        and amplitude > 1e-4
        and waveform_correction < 0.25
    )
    return DDEOrbitRefinement(
        history_coefficients=coefficients,
        period=period,
        initial_history=initial_history,
        endpoint_history=endpoint_history,
        optimizer_success=bool(result.success),
        converged=converged,
        nfev=int(result.nfev),
        cost=float(result.cost),
        relative_closure_defect=relative_defect,
        time_phase_defect=time_phase,
        relative_history_correction=history_correction,
        relative_waveform_correction=waveform_correction,
        equation_rms_residual=equation_rms,
        equation_max_residual=equation_max,
        amplitude=amplitude,
        message=str(result.message),
    )


def periodic_orbit_record(prefix, refinement):
    return {
        f"{prefix}_optimizer_success": refinement.optimizer_success,
        f"{prefix}_converged": refinement.converged,
        f"{prefix}_period": refinement.period,
        f"{prefix}_closure_defect": refinement.relative_closure_defect,
        f"{prefix}_time_phase_defect": refinement.time_phase_defect,
        f"{prefix}_state_correction": refinement.relative_state_correction,
        f"{prefix}_speed_norm": refinement.speed_norm,
        f"{prefix}_nfev": refinement.nfev,
        f"{prefix}_cost": refinement.cost,
        f"{prefix}_message": refinement.message,
    }


def dde_orbit_record(prefix, refinement):
    return {
        f"{prefix}_optimizer_success": refinement.optimizer_success,
        f"{prefix}_converged": refinement.converged,
        f"{prefix}_period": refinement.period,
        f"{prefix}_closure_defect": refinement.relative_closure_defect,
        f"{prefix}_time_phase_defect": refinement.time_phase_defect,
        f"{prefix}_history_correction": refinement.relative_history_correction,
        f"{prefix}_waveform_correction": refinement.relative_waveform_correction,
        f"{prefix}_equation_rms_residual": refinement.equation_rms_residual,
        f"{prefix}_equation_max_residual": refinement.equation_max_residual,
        f"{prefix}_amplitude": refinement.amplitude,
        f"{prefix}_nfev": refinement.nfev,
        f"{prefix}_cost": refinement.cost,
        f"{prefix}_message": refinement.message,
    }


### 13.2 Opt-in orbit search, collocation correction, and validation


In [ ]:
def generate_learned_attractor_trajectory(
    model,
    initial_state,
    dt,
    burn_time=500.0,
    sample_time=3000.0,
):
    times = np.arange(0.0, burn_time + sample_time + dt, dt)
    trajectory = safe_rk4_rollout(model, initial_state, times)
    if trajectory is None:
        raise RuntimeError("Learned-model orbit-search rollout diverged.")
    burn_steps = int(round(burn_time / dt))
    return trajectory[burn_steps:]


ORBIT_PROPOSAL_TRAINING_WINDOWS = [
    (117.0, 417.0),
    (700.0, 1000.0),
    (1100.0, 1400.0),
    (2700.0, 3000.0),
    (3100.0, 3400.0),
]
ORBIT_PROPOSAL_RIDGE_ALPHA = 5e-5
ORBIT_PROPOSAL_PERIOD_BAND = (15.0, 80.0)


def build_orbit_polynomial_exponents(state_dimension, max_degree):
    """Build one exponent array without relying on later notebook helpers."""
    exponents = [np.zeros(state_dimension, dtype=int)]
    for degree in range(1, max_degree + 1):
        for combo in combinations_with_replacement(range(state_dimension), degree):
            exponent = np.zeros(state_dimension, dtype=int)
            for index in combo:
                exponent[index] += 1
            exponents.append(exponent)
    return np.asarray(exponents)


def fit_orbit_proposal_model(training_windows):
    """Fit the same polynomial model class on several disjoint orbit windows."""
    powers = build_orbit_polynomial_exponents(
        EMBEDDING_DIMENSION, BASELINE_POLY_DEGREE
    )
    library = PolynomialLibraryAdapter(powers)
    theta_blocks = []
    derivative_blocks = []
    delay_steps = int(round(EMBEDDING_DELAY / DT))
    for start_time, end_time in training_windows:
        embedded = build_delay_embedding(
            data,
            int(round(start_time / DT)),
            int(round(end_time / DT)),
            delay_steps,
            EMBEDDING_DIMENSION,
        )
        theta_blocks.append(library.transform(embedded))
        derivative_blocks.append(
            np.gradient(embedded, DT, axis=0, edge_order=2)
        )

    theta = np.vstack(theta_blocks)
    derivatives = np.vstack(derivative_blocks)
    scales = np.sqrt(np.mean(theta**2, axis=0))
    scales[scales < 1e-14] = 1.0
    theta_normalized = theta / scales
    feature_degree = powers.sum(axis=1)
    penalty = np.maximum(feature_degree, 1) ** RIDGE_DEGREE_WEIGHT
    penalty[0] = 0.0
    normal_matrix = (
        theta_normalized.T @ theta_normalized
        + ORBIT_PROPOSAL_RIDGE_ALPHA * np.diag(penalty)
    )
    rhs = theta_normalized.T @ derivatives
    coefficients_normalized = np.linalg.solve(normal_matrix, rhs).T
    coefficients = coefficients_normalized / scales[None, :]
    return FastPolynomialModel(
        NormalizedRidgePolynomialModel(coefficients, powers)
    )


if RUN_PERIODIC_ORBIT_SEARCH:
    orbit_proposal_model = fit_orbit_proposal_model(
        ORBIT_PROPOSAL_TRAINING_WINDOWS
    )
    learned_attractor = generate_learned_attractor_trajectory(
        orbit_proposal_model, X_test[1000], DT
    )
    periodic_candidates = scan_periodic_recurrences(
        learned_attractor,
        DT,
        min_period=15.0,
        max_period=260.0,
        search_stride=5,
        neighbors=240,
        max_candidates=80,
    )
    periodic_candidates.to_csv(
        OUTPUT_DIR / "mg_periodic_recurrence_candidates.csv", index=False
    )
    if periodic_candidates.empty:
        raise RuntimeError("No periodic-recurrence seeds were found.")
    print("Top periodic-recurrence seeds:")
    display(periodic_candidates.head(10))

    learned_rhs = lambda state: model_rhs(orbit_proposal_model, state)
    learned_endpoint = lambda state, period: adaptive_endpoint(
        state,
        period,
        learned_rhs,
        rtol=1e-9,
        atol=1e-11,
        max_step=0.1,
        max_abs=3.0,
    )

    orbit_rows = []
    resolution_rows = []
    learned_orbit_states = []
    seed_history_coefficients_all = []
    true_history_coefficients = []
    # Shorter orbits suppress long-time shooting amplification and are the
    # appropriate first transfer target for the true-DDE/CAP pipeline.
    proposal_candidates = periodic_candidates[
        (periodic_candidates["period"] >= ORBIT_PROPOSAL_PERIOD_BAND[0])
        & (periodic_candidates["period"] < ORBIT_PROPOSAL_PERIOD_BAND[1])
    ].head(1)
    if proposal_candidates.empty:
        raise RuntimeError("No recurrence was found in the proposal period band.")
    for candidate_index, candidate in proposal_candidates.iterrows():
        reference_state = learned_attractor[int(candidate["start_index"])]
        learned_refinement = refine_periodic_orbit(
            reference_state,
            candidate["period"],
            learned_endpoint,
            learned_rhs,
            max_nfev=80,
            defect_tolerance=1e-8,
            state_bounds=(STATE_BOUNDS[0], STATE_BOUNDS[1]),
        )

        orbit_times, orbit_signal = sample_learned_periodic_orbit(
            learned_refinement.state,
            learned_refinement.period,
            learned_rhs,
            n_samples=4096,
            max_step=0.03,
        )
        seed_history_coefficients = fit_periodic_history(
            orbit_times,
            orbit_signal,
            learned_refinement.period,
            harmonics=200,
        )
        seed_equation_residual = mackey_glass_equation_residual(
            seed_history_coefficients,
            learned_refinement.period,
            BETA,
            GAMMA,
            NONLINEARITY,
            DDE_DELAY,
            n_nodes=4096,
        )
        seed_equation_rms = float(
            np.sqrt(np.mean(seed_equation_residual**2))
        )
        seed_derivative_rms = float(
            np.sqrt(
                np.mean(
                    evaluate_periodic_history_derivative(
                        orbit_times,
                        seed_history_coefficients,
                        learned_refinement.period,
                    )
                    ** 2
                )
            )
        )
        seed_equation_relative = float(
            seed_equation_rms / max(seed_derivative_rms, 1e-12)
        )
        _, _, true_defect_before = mackey_glass_history_closure(
            seed_history_coefficients,
            learned_refinement.period,
            BETA,
            GAMMA,
            NONLINEARITY,
            DDE_DELAY,
            history_nodes=129,
            max_step=0.05,
        )
        true_refinement = None
        for collocation_harmonics in [40, 80, 120, 160, 200]:
            resolution_refinement = refine_mackey_glass_periodic_orbit(
                seed_history_coefficients,
                learned_refinement.period,
                BETA,
                GAMMA,
                NONLINEARITY,
                DDE_DELAY,
                history_nodes=129,
                max_step=0.05,
                max_nfev=100,
                defect_tolerance=1e-9,
                collocation_harmonics=collocation_harmonics,
                collocation_oversampling=4,
            )
            resolution_rows.append(
                {
                    "candidate": int(candidate_index),
                    "collocation_harmonics": collocation_harmonics,
                    "period": resolution_refinement.period,
                    "equation_rms_residual": (
                        resolution_refinement.equation_rms_residual
                    ),
                    "equation_max_residual": (
                        resolution_refinement.equation_max_residual
                    ),
                    "independent_closure_defect": (
                        resolution_refinement.relative_closure_defect
                    ),
                    "relative_waveform_correction": (
                        resolution_refinement.relative_waveform_correction
                    ),
                    "accepted_at_resolution": resolution_refinement.converged,
                }
            )
            true_refinement = resolution_refinement

        learned_validation_endpoint = adaptive_endpoint(
            learned_refinement.state,
            learned_refinement.period,
            learned_rhs,
            rtol=1e-11,
            atol=1e-13,
            max_step=0.025,
            max_abs=3.0,
        )
        learned_validation_defect = periodic_closure_defect(
            learned_refinement.state, learned_validation_endpoint
        )
        _, _, true_validation_defect = mackey_glass_history_closure(
            true_refinement.history_coefficients,
            true_refinement.period,
            BETA,
            GAMMA,
            NONLINEARITY,
            DDE_DELAY,
            history_nodes=257,
            max_step=0.0125,
            rtol=2e-13,
            atol=2e-15,
        )

        row = {
            "candidate": int(candidate_index),
            "proposal_training_window_count": len(
                ORBIT_PROPOSAL_TRAINING_WINDOWS
            ),
            "proposal_ridge_alpha": ORBIT_PROPOSAL_RIDGE_ALPHA,
            "seed_period": float(candidate["period"]),
            "seed_relative_return_error": float(
                candidate["relative_return_error"]
            ),
            **periodic_orbit_record("learned", learned_refinement),
            "learned_validation_defect": learned_validation_defect,
            "learned_validated_converged": bool(
                learned_refinement.converged
                and learned_validation_defect < 1e-8
            ),
            "true_defect_before_correction": true_defect_before,
            "true_equation_rms_before_correction": seed_equation_rms,
            "true_equation_relative_before_correction": seed_equation_relative,
            **dde_orbit_record("true", true_refinement),
            "true_validation_defect": true_validation_defect,
            "true_validated_converged": bool(
                true_refinement.converged
                and true_validation_defect < 1e-9
            ),
            "transfer_period_correction": float(
                true_refinement.period - learned_refinement.period
            ),
        }
        orbit_rows.append(row)
        learned_orbit_states.append(learned_refinement.state)
        seed_history_coefficients_all.append(seed_history_coefficients)
        true_history_coefficients.append(
            true_refinement.history_coefficients
        )
        print(
            f"candidate {candidate_index}: "
            f"seed={candidate['relative_return_error']:.3e}, "
            f"learned={learned_refinement.relative_closure_defect:.3e}, "
            f"true-before-shooting={true_defect_before:.3e}, "
            f"true-before-equation={seed_equation_relative:.3e}, "
            f"true-after={true_refinement.relative_closure_defect:.3e}, "
            f"validated={true_validation_defect:.3e}, "
            f"accepted={row['true_validated_converged']}"
        )

    orbit_results = pd.DataFrame(orbit_rows)
    orbit_results.to_csv(
        OUTPUT_DIR / "mg_orbit_transfer_pilot.csv", index=False
    )
    pd.DataFrame(
        ORBIT_PROPOSAL_TRAINING_WINDOWS,
        columns=["start_time", "end_time"],
    ).to_csv(
        OUTPUT_DIR / "mg_orbit_proposal_training_windows.csv", index=False
    )
    resolution_results = pd.DataFrame(resolution_rows)
    resolution_results.to_csv(
        OUTPUT_DIR / "mg_true_dde_resolution_continuation.csv", index=False
    )
    print("True-DDE Fourier-resolution continuation:")
    display(resolution_results)
    np.savez(
        OUTPUT_DIR / "mg_orbit_transfer_pilot.npz",
        learned_states=np.asarray(learned_orbit_states),
        seed_history_coefficients=np.asarray(seed_history_coefficients_all),
        true_history_coefficients=np.asarray(true_history_coefficients),
        learned_periods=orbit_results["learned_period"].to_numpy(),
        true_periods=orbit_results["true_period"].to_numpy(),
    )

    figure, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    axis = axes[0]
    best_row = orbit_results.iloc[0]
    stages = [
        "Recurrence\nseed",
        "Refined\nlearned orbit",
        "True DDE\nbefore correction",
        "True DDE\nafter correction",
    ]
    stage_defects = [
        best_row["seed_relative_return_error"],
        best_row["learned_validation_defect"],
        best_row["true_defect_before_correction"],
        best_row["true_validation_defect"],
    ]
    axis.semilogy(range(len(stages)), stage_defects, "o-", lw=1.4, ms=7)
    axis.set_xticks(range(len(stages)), stages)
    axis.set(
        ylabel="Relative closure defect",
        title="Mackey–Glass learned-to-true periodic-orbit pilot",
    )

    phase = np.linspace(0.0, 1.0, 2048, endpoint=False)
    learned_waveform = evaluate_periodic_history(
        phase * best_row["learned_period"],
        seed_history_coefficients_all[0],
        best_row["learned_period"],
    )
    true_waveform = evaluate_periodic_history(
        phase * best_row["true_period"],
        true_history_coefficients[0],
        best_row["true_period"],
    )
    axes[1].plot(phase, learned_waveform, "--", lw=1.2, label="Learned orbit seed")
    axes[1].plot(phase, true_waveform, "-", lw=1.2, label="Corrected true-DDE orbit")
    axes[1].set(
        xlabel="Cycle phase",
        ylabel="$x$",
        title=(
            "Transferred waveform "
            f"({best_row['true_waveform_correction']:.1%} correction)"
        ),
    )
    axes[1].legend()
    figure.tight_layout()
    save_figure(figure, "MG_Orbit_Transfer_Pilot.png", dpi=250)
    plt.show()
else:
    print(
        "Periodic-orbit search skipped. "
        "Set RUN_PERIODIC_ORBIT_SEARCH = True to run."
    )


## Notes for repository maintainers

- This notebook is a chaos-aware derivative of `MG_FINAL_CLEAN.ipynb`; the source file remains untouched.
- The fit/validation/test/attractor-reference windows are disjoint to prevent leakage.
- Ridge is applied after RMS feature normalization, with a mild degree-weighted penalty.
- The default model was selected by short-term validation plus long-run attractor matching, then confirmed by multi-IC Benettin MLE estimates.
- Optional legacy pruning, library-comparison, manifold, and noise sections remain available but are disabled by default.
- Periodic-return candidates are only seeds; report an orbit only when the learned and true-system correction plus independent closure validation have passed.
- Long-term pointwise agreement is not a valid success criterion for a chaotic system. Use invariant statistics, spectra, boundedness, and Lyapunov estimates after the forecast horizon.


## Poster figure: true and approximated periodic orbits in 3D

Both curves are plotted in the same physical delay-coordinate space, $(x(t),x(t-\tau),x(t-2\tau))$, using the true Mackey–Glass delay $\tau=17$.


In [ ]:
%matplotlib widget
# Poster-ready 3D comparison of the validated true-DDE orbit and learned approximation.
orbit_archive_path = OUTPUT_DIR / "mg_orbit_transfer_pilot.npz"
if not orbit_archive_path.exists():
    raise FileNotFoundError(
        "Run Section 13.2 first to create mg_orbit_transfer_pilot.npz."
    )

with np.load(orbit_archive_path) as orbit_archive:
    approximated_coefficients = orbit_archive["seed_history_coefficients"][0]
    actual_coefficients = orbit_archive["true_history_coefficients"][0]
    approximated_period = float(orbit_archive["learned_periods"][0])
    actual_period = float(orbit_archive["true_periods"][0])

def periodic_delay_coordinates(coefficients, period, delay, n_points=4000):
    """Sample one closed orbit in (x(t), x(t-delay), x(t-2*delay))."""
    times = np.linspace(0.0, period, n_points, endpoint=True)
    return np.column_stack(
        [
            evaluate_periodic_history(times - shift, coefficients, period)
            for shift in (0.0, delay, 2.0 * delay)
        ]
    )

approximated_orbit_3d = periodic_delay_coordinates(
    approximated_coefficients, approximated_period, DDE_DELAY
)
actual_orbit_3d = periodic_delay_coordinates(
    actual_coefficients, actual_period, DDE_DELAY
)

with plt.rc_context(
    {
        "font.size": 13,
        "axes.titlesize": 18,
        "axes.labelsize": 15,
        "legend.fontsize": 12,
        "grid.alpha": 0.18,
    }
):
    figure = plt.figure(figsize=(9.2, 8.2), facecolor="white")
    axis = figure.add_subplot(111, projection="3d")

    # Draw the true orbit first and the dashed approximation on top so that
    # their near-perfect agreement remains visible wherever they overlap.
    axis.plot(
        actual_orbit_3d[:, 0],
        actual_orbit_3d[:, 1],
        actual_orbit_3d[:, 2],
        color="#0072B2",
        lw=3.2,
        alpha=0.96,
        label="True Mackey–Glass DDE",
    )
    axis.plot(
        approximated_orbit_3d[:, 0],
        approximated_orbit_3d[:, 1],
        approximated_orbit_3d[:, 2],
        color="#D55E00",
        lw=2.0,
        ls=(0, (4, 2)),
        alpha=1.0,
        label="Learned approximation",
    )

    axis.set(
        xlabel=r"$x(t)$",
        ylabel=rf"$x(t-\tau)$,  $\tau={DDE_DELAY:g}$",
        zlabel=r"$x(t-2\tau)$",
        title="Mackey–Glass Periodic Orbit",
    )
    axis.view_init(elev=24, azim=-54)

    combined_orbits = np.vstack([actual_orbit_3d, approximated_orbit_3d])
    coordinate_ranges = np.ptp(combined_orbits, axis=0)
    coordinate_ranges[coordinate_ranges < 1e-12] = 1.0
    axis.set_box_aspect(coordinate_ranges)
    for dimension, setter in enumerate(
        (axis.set_xlim, axis.set_ylim, axis.set_zlim)
    ):
        lower = combined_orbits[:, dimension].min()
        upper = combined_orbits[:, dimension].max()
        padding = 0.06 * (upper - lower)
        setter(lower - padding, upper + padding)

    axis.xaxis.pane.set_facecolor((1.0, 1.0, 1.0, 0.0))
    axis.yaxis.pane.set_facecolor((1.0, 1.0, 1.0, 0.0))
    axis.zaxis.pane.set_facecolor((1.0, 1.0, 1.0, 0.0))
    axis.legend(loc="upper left", frameon=True, framealpha=0.94)
    figure.tight_layout()

    poster_png = OUTPUT_DIR / "MG_Periodic_Orbit_3D_Comparison.png"
    figure.savefig(
        poster_png, dpi=600, bbox_inches="tight", facecolor="white"
    )
    plt.show()

print(f"Saved poster figure to: {poster_png.resolve()}")
